In [1]:
# 1. HÜCRE — Gerekli kütüphaneler ve çalışma ortamı kontrolü

%pip install -q scikit-image pyyaml joblib

import sys
import torch
import numpy as np
import pandas as pd
import sklearn
import skimage
import joblib
import yaml

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA kullanılabilir:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("UYARI: GPU bulunamadı. Çalışma zamanı türünden T4 GPU seç.")

Python: 3.12.13
PyTorch: 2.11.0+cu128
CUDA kullanılabilir: True
GPU: Tesla T4


In [2]:
# 2. HÜCRE — Google Drive bağlantısı

from google.colab import drive

drive.mount("/content/drive", force_remount=False)

print("Google Drive başarıyla bağlandı.")

Mounted at /content/drive
Google Drive başarıyla bağlandı.


In [3]:
# 3. HÜCRE — Ağız verisi ve metadata yollarının kontrolü

from pathlib import Path

DATA_ROOT = Path(
    "/content/drive/MyDrive/AISC DeepFake Çalışmaları/"
    "Deneyler/Dilara/Deney 1/Ağız/mouth_roi_output"
)

EXPERIMENTS_ROOT = DATA_ROOT.parents[3]

ROI_METADATA_PATH = DATA_ROOT / "metadata.csv"

SELECTION_METADATA_PATH = (
    EXPERIMENTS_ROOT
    / "Deney 1 Frame"
    / "secim_metadata.csv"
)

RESULTS_ROOT = (
    DATA_ROOT.parent
    / "Sonuçlar"
    / "VGG16_HOG_GIST_RBF_SVM"
)

print("Ağız klasörü:", DATA_ROOT)
print("Ağız klasörü bulundu:", DATA_ROOT.exists())

print("\nROI metadata:", ROI_METADATA_PATH)
print("ROI metadata bulundu:", ROI_METADATA_PATH.exists())

print("\nFrame seçim metadata:", SELECTION_METADATA_PATH)
print("Frame seçim metadata bulundu:", SELECTION_METADATA_PATH.exists())

if DATA_ROOT.exists():
    files = list(DATA_ROOT.iterdir())

    print("\nKlasördeki ilk dosya ve klasörler:")
    for path in sorted(files)[:20]:
        symbol = "📁" if path.is_dir() else "📄"
        print(symbol, path.name)

    image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

    image_count = sum(
        1
        for path in DATA_ROOT.rglob("*")
        if path.is_file() and path.suffix.lower() in image_extensions
    )

    print("\nBulunan ağız görseli sayısı:", image_count)

Ağız klasörü: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Dilara/Deney 1/Ağız/mouth_roi_output
Ağız klasörü bulundu: True

ROI metadata: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Dilara/Deney 1/Ağız/mouth_roi_output/metadata.csv
ROI metadata bulundu: True

Frame seçim metadata: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Deney 1 Frame/secim_metadata.csv
Frame seçim metadata bulundu: True

Klasördeki ilk dosya ve klasörler:
📁 fake
📄 final_validation_report.json
📄 final_validation_report.txt
📄 metadata.csv
📄 processing_summary.json
📁 real
📄 run_config.json

Bulunan ağız görseli sayısı: 5974


In [4]:
# 4. HÜCRE — Metadata dosyalarını okuma ve ilk yapısal kontrol

import pandas as pd

if not ROI_METADATA_PATH.exists():
    raise FileNotFoundError(
        f"Ağız metadata dosyası bulunamadı:\n{ROI_METADATA_PATH}"
    )

if not SELECTION_METADATA_PATH.exists():
    raise FileNotFoundError(
        f"Frame seçim metadata dosyası bulunamadı:\n"
        f"{SELECTION_METADATA_PATH}"
    )

roi_metadata = pd.read_csv(ROI_METADATA_PATH)
selection_metadata = pd.read_csv(SELECTION_METADATA_PATH)

print("AĞIZ ROI METADATA")
print("-" * 60)
print("Satır sayısı:", len(roi_metadata))
print("Sütunlar:")
print(roi_metadata.columns.tolist())

print("\nİlk 3 satır:")
display(roi_metadata.head(3))

print("\nFRAME SEÇİM METADATA")
print("-" * 60)
print("Satır sayısı:", len(selection_metadata))
print("Sütunlar:")
print(selection_metadata.columns.tolist())

print("\nİlk 3 satır:")
display(selection_metadata.head(3))

print("\nSPLIT DAĞILIMI")
print("-" * 60)

if "split" in roi_metadata.columns:
    print(
        roi_metadata["split"]
        .astype(str)
        .str.lower()
        .value_counts(dropna=False)
    )
else:
    print("UYARI: Ağız metadata dosyasında 'split' sütunu bulunamadı.")

print("\nETİKET DAĞILIMI")
print("-" * 60)

if "label" in roi_metadata.columns:
    print(
        roi_metadata["label"]
        .astype(str)
        .str.lower()
        .value_counts(dropna=False)
    )
else:
    print("UYARI: Ağız metadata dosyasında 'label' sütunu bulunamadı.")

AĞIZ ROI METADATA
------------------------------------------------------------
Satır sayısı: 3098
Sütunlar:
['sample_id', 'source_frame', 'relative_frame_path', 'label', 'split', 'video_id', 'frame_stem', 'face_id', 'face_id_scope', 'faces_in_frame', 'mouth_detected', 'mouth_path', 'debug_path', 'landmarks_path', 'mouth_bbox_x1', 'mouth_bbox_y1', 'mouth_bbox_x2', 'mouth_bbox_y2', 'mouth_width_px', 'mouth_height_px', 'landmark_count', 'status', 'error', 'processing_ms']

İlk 3 satır:


,sample_id,source_frame,relative_frame_path,label,split,video_id,frame_stem,face_id,face_id_scope,faces_in_frame,...,mouth_bbox_x1,mouth_bbox_y1,mouth_bbox_x2,mouth_bbox_y2,mouth_width_px,mouth_height_px,landmark_count,status,error,processing_ms
0,fake_train_01130_face00,/content/drive/MyDrive/AISC Çalışmalar/Deney...,Fake/train/fake_train_01130.jpg,fake,train,fake_train_01130,fake_train_01130,0.0,frame,1,...,1275.0,370.0,1409.0,461.0,134.0,91.0,40,SUCCESS,NaN,395.327
1,fake_train_fake_train_01130_face00,/content/drive/MyDrive/AISC Çalışmalar/Deney...,Fake/train/fake_train_01130.jpg,fake,train,fake_train_01130,fake_train_01130,0.0,frame,1,...,1275.0,370.0,1409.0,461.0,134.0,91.0,40,SUCCESS,NaN,70.473
2,fake_train_fake_train_00000_face00,/content/drive/MyDrive/AISC Çalışmalar/Deney...,Fake/train/fake_train_00000.jpg,fake,train,fake_train_00000,fake_train_00000,0.0,frame,1,...,485.0,452.0,653.0,514.0,168.0,62.0,40,SUCCESS,NaN,56.912



FRAME SEÇİM METADATA
------------------------------------------------------------
Satır sayısı: 3000
Sütunlar:
['sinif', 'split', 'orijinal_yol', 'yeni_yol', 'dosya_adi']

İlk 3 satır:


,sinif,split,orijinal_yol,yeni_yol,dosya_adi
0,Real,train,/content/drive/MyDrive/AISC DeepFake Çalışmala...,/content/drive/MyDrive/AISC DeepFake Çalışmala...,real_train_00000.jpg
1,Real,train,/content/drive/MyDrive/AISC DeepFake Çalışmala...,/content/drive/MyDrive/AISC DeepFake Çalışmala...,real_train_00001.jpg
2,Real,train,/content/drive/MyDrive/AISC DeepFake Çalışmala...,/content/drive/MyDrive/AISC DeepFake Çalışmala...,real_train_00002.jpg



SPLIT DAĞILIMI
------------------------------------------------------------
split
train    2479
test      310
val       309
Name: count, dtype: int64

ETİKET DAĞILIMI
------------------------------------------------------------
label
fake    1567
real    1531
Name: count, dtype: int64


In [5]:
# 5. HÜCRE — Gerçek ağız ROI sayısı ve tekrarlanan kayıtların kontrolü

from pathlib import Path
import pandas as pd

roi_check = roi_metadata.copy()
selection_check = selection_metadata.copy()

# Her iki metadata dosyasında ortak eşleştirme anahtarı oluştur
roi_check["file_name"] = roi_check["relative_frame_path"].map(
    lambda value: Path(str(value)).name
)

selection_check["file_name"] = (
    selection_check["dosya_adi"]
    .astype(str)
    .map(lambda value: Path(value).name)
)

# mouth_detected değerini güvenli şekilde Boolean'a dönüştür
roi_check["mouth_detected_bool"] = (
    roi_check["mouth_detected"]
    .astype(str)
    .str.strip()
    .str.lower()
    .isin(["true", "1", "yes"])
)

print("FRAME SEÇİM KONTROLÜ")
print("-" * 60)
print("Seçilen toplam frame:", len(selection_check))
print(
    "Benzersiz seçilen dosya adı:",
    selection_check["file_name"].nunique()
)

print("\nAĞIZ METADATA KONTROLÜ")
print("-" * 60)
print("Metadata toplam satır:", len(roi_check))
print(
    "Benzersiz frame/dosya adı:",
    roi_check["file_name"].nunique()
)
print(
    "Tekrarlanan metadata satırı:",
    roi_check.duplicated("file_name", keep=False).sum()
)
print(
    "Fazladan kayıt sayısı:",
    len(roi_check) - roi_check["file_name"].nunique()
)

print("\nAĞIZ TESPİT DURUMU")
print("-" * 60)
print(roi_check["mouth_detected_bool"].value_counts(dropna=False))

successful_rows = roi_check[
    roi_check["mouth_detected_bool"]
].copy()

print("\nAğız tespit edilen metadata satırı:", len(successful_rows))
print(
    "Ağız tespit edilen benzersiz frame:",
    successful_rows["file_name"].nunique()
)

if "mouth_path" in successful_rows.columns:
    print(
        "Benzersiz mouth_path:",
        successful_rows["mouth_path"].dropna().nunique()
    )

if "debug_path" in successful_rows.columns:
    print(
        "Benzersiz debug_path:",
        successful_rows["debug_path"].dropna().nunique()
    )

# Seçim metadata'sında olup ağız metadata'sında olmayan frameler
selected_names = set(selection_check["file_name"])
roi_names = set(roi_check["file_name"])

missing_from_roi_metadata = sorted(selected_names - roi_names)
unexpected_roi_records = sorted(roi_names - selected_names)

print("\nEŞLEŞTİRME KONTROLÜ")
print("-" * 60)
print(
    "Seçilmiş fakat ROI metadata'da bulunmayan:",
    len(missing_from_roi_metadata)
)
print(
    "Seçim listesinde bulunmayan fazladan isim:",
    len(unexpected_roi_records)
)

print("\nTekrarlanan ilk 10 frame:")
duplicate_preview = (
    roi_check[
        roi_check.duplicated("file_name", keep=False)
    ][
        [
            "file_name",
            "sample_id",
            "label",
            "split",
            "mouth_detected",
            "mouth_path",
        ]
    ]
    .sort_values("file_name")
    .head(20)
)

display(duplicate_preview)

FRAME SEÇİM KONTROLÜ
------------------------------------------------------------
Seçilen toplam frame: 3000
Benzersiz seçilen dosya adı: 3000

AĞIZ METADATA KONTROLÜ
------------------------------------------------------------
Metadata toplam satır: 3098
Benzersiz frame/dosya adı: 3000
Tekrarlanan metadata satırı: 192
Fazladan kayıt sayısı: 98

AĞIZ TESPİT DURUMU
------------------------------------------------------------
mouth_detected_bool
True     2987
False     111
Name: count, dtype: int64

Ağız tespit edilen metadata satırı: 2987
Ağız tespit edilen benzersiz frame: 2889
Benzersiz mouth_path: 2987
Benzersiz debug_path: 2987

EŞLEŞTİRME KONTROLÜ
------------------------------------------------------------
Seçilmiş fakat ROI metadata'da bulunmayan: 0
Seçim listesinde bulunmayan fazladan isim: 0

Tekrarlanan ilk 10 frame:


,file_name,sample_id,label,split,mouth_detected,mouth_path
1429,fake_test_00021.jpg,fake_test_fake_test_00021_face00,fake,test,True,/content/drive/MyDrive/AISC Çalışmalar/Deney...
1430,fake_test_00021.jpg,fake_test_fake_test_00021_face01,fake,test,True,/content/drive/MyDrive/AISC Çalışmalar/Deney...
1457,fake_test_00047.jpg,fake_test_fake_test_00047_face01,fake,test,True,/content/drive/MyDrive/AISC Çalışmalar/Deney...
1456,fake_test_00047.jpg,fake_test_fake_test_00047_face00,fake,test,True,/content/drive/MyDrive/AISC Çalışmalar/Deney...
1465,fake_test_00054.jpg,fake_test_fake_test_00054_face01,fake,test,True,/content/drive/MyDrive/AISC Çalışmalar/Deney...
1464,fake_test_00054.jpg,fake_test_fake_test_00054_face00,fake,test,True,/content/drive/MyDrive/AISC Çalışmalar/Deney...
1486,fake_test_00074.jpg,fake_test_fake_test_00074_face01,fake,test,True,/content/drive/MyDrive/AISC Çalışmalar/Deney...
1485,fake_test_00074.jpg,fake_test_fake_test_00074_face00,fake,test,True,/content/drive/MyDrive/AISC Çalışmalar/Deney...
1488,fake_test_00075.jpg,fake_test_fake_test_00075_face01,fake,test,True,/content/drive/MyDrive/AISC Çalışmalar/Deney...
1487,fake_test_00075.jpg,fake_test_fake_test_00075_face00,fake,test,True,/content/drive/MyDrive/AISC Çalışmalar/Deney...


In [6]:
# 6. HÜCRE — Birden fazla yüz bulunan framelerde ana yüz seçimi için sütun kontrolü

print("Metadata sütunlarının tamamı:")
for index, column in enumerate(roi_check.columns, start=1):
    print(f"{index:02d}. {column}")

duplicate_face_rows = roi_check[
    roi_check["mouth_detected_bool"]
    & roi_check.duplicated("file_name", keep=False)
].copy()

print("\nBirden fazla ağız bulunan benzersiz frame sayısı:")
print(duplicate_face_rows["file_name"].nunique())

# Yüz ve ağız kutularıyla ilgili sütunları bul
bbox_columns = [
    column
    for column in roi_check.columns
    if any(
        keyword in column.lower()
        for keyword in ["bbox", "width", "height", "area"]
    )
]

print("\nKutu/boyutla ilgili sütunlar:")
print(bbox_columns)

columns_to_show = [
    column
    for column in [
        "file_name",
        "sample_id",
        "face_id",
        "faces_in_frame",
        "mouth_detected",
        *bbox_columns,
        "mouth_path",
    ]
    if column in duplicate_face_rows.columns
]

print("\nÇok yüzlü ilk 5 frame:")
first_multi_face_names = (
    duplicate_face_rows["file_name"]
    .drop_duplicates()
    .head(5)
)

display(
    duplicate_face_rows[
        duplicate_face_rows["file_name"].isin(first_multi_face_names)
    ][columns_to_show]
    .sort_values(["file_name", "face_id"])
)

Metadata sütunlarının tamamı:
01. sample_id
02. source_frame
03. relative_frame_path
04. label
05. split
06. video_id
07. frame_stem
08. face_id
09. face_id_scope
10. faces_in_frame
11. mouth_detected
12. mouth_path
13. debug_path
14. landmarks_path
15. mouth_bbox_x1
16. mouth_bbox_y1
17. mouth_bbox_x2
18. mouth_bbox_y2
19. mouth_width_px
20. mouth_height_px
21. landmark_count
22. status
23. error
24. processing_ms
25. file_name
26. mouth_detected_bool

Birden fazla ağız bulunan benzersiz frame sayısı:
94

Kutu/boyutla ilgili sütunlar:
['mouth_bbox_x1', 'mouth_bbox_y1', 'mouth_bbox_x2', 'mouth_bbox_y2', 'mouth_width_px', 'mouth_height_px']

Çok yüzlü ilk 5 frame:


,file_name,sample_id,face_id,faces_in_frame,mouth_detected,mouth_bbox_x1,mouth_bbox_y1,mouth_bbox_x2,mouth_bbox_y2,mouth_width_px,mouth_height_px,mouth_path
30,fake_train_00028.jpg,fake_train_fake_train_00028_face00,0.0,2,True,414.0,425.0,547.0,483.0,133.0,58.0,/content/drive/MyDrive/AISC Çalışmalar/Deney...
31,fake_train_00028.jpg,fake_train_fake_train_00028_face01,1.0,2,True,1055.0,342.0,1145.0,412.0,90.0,70.0,/content/drive/MyDrive/AISC Çalışmalar/Deney...
36,fake_train_00033.jpg,fake_train_fake_train_00033_face00,0.0,2,True,731.0,390.0,810.0,426.0,79.0,36.0,/content/drive/MyDrive/AISC Çalışmalar/Deney...
37,fake_train_00033.jpg,fake_train_fake_train_00033_face01,1.0,2,True,1199.0,548.0,1295.0,584.0,96.0,36.0,/content/drive/MyDrive/AISC Çalışmalar/Deney...
76,fake_train_00072.jpg,fake_train_fake_train_00072_face00,0.0,2,True,521.0,391.0,648.0,462.0,127.0,71.0,/content/drive/MyDrive/AISC Çalışmalar/Deney...
77,fake_train_00072.jpg,fake_train_fake_train_00072_face01,1.0,2,True,1227.0,321.0,1306.0,381.0,79.0,60.0,/content/drive/MyDrive/AISC Çalışmalar/Deney...
151,fake_train_00146.jpg,fake_train_fake_train_00146_face00,0.0,2,True,398.0,180.0,460.0,217.0,62.0,37.0,/content/drive/MyDrive/AISC Çalışmalar/Deney...
152,fake_train_00146.jpg,fake_train_fake_train_00146_face01,1.0,2,True,196.0,155.0,239.0,172.0,43.0,17.0,/content/drive/MyDrive/AISC Çalışmalar/Deney...
0,fake_train_01130.jpg,fake_train_01130_face00,0.0,1,True,1275.0,370.0,1409.0,461.0,134.0,91.0,/content/drive/MyDrive/AISC Çalışmalar/Deney...
1,fake_train_01130.jpg,fake_train_fake_train_01130_face00,0.0,1,True,1275.0,370.0,1409.0,461.0,134.0,91.0,/content/drive/MyDrive/AISC Çalışmalar/Deney...


In [7]:
# 7. HÜCRE — Her frame için tek ve ana ağız ROI kaydını seçme

import numpy as np
import pandas as pd
from pathlib import Path

primary_candidates = roi_check[
    roi_check["mouth_detected_bool"]
].copy()

# Sayısal sütunları güvenli biçimde dönüştür
for column in [
    "face_id",
    "mouth_width_px",
    "mouth_height_px",
    "mouth_bbox_x1",
    "mouth_bbox_y1",
    "mouth_bbox_x2",
    "mouth_bbox_y2",
]:
    primary_candidates[column] = pd.to_numeric(
        primary_candidates[column],
        errors="coerce",
    )

# Ağız kutusunun alanını hesapla
primary_candidates["mouth_area_px"] = (
    primary_candidates["mouth_width_px"]
    * primary_candidates["mouth_height_px"]
)

if primary_candidates["mouth_area_px"].isna().any():
    raise ValueError(
        "Ağız alanı hesaplanamayan başarılı kayıtlar bulundu."
    )

# mouth_path içindeki eski Drive yolu çalışmazsa,
# gerçek DATA_ROOT üzerinden dosya yolunu yeniden kur
def resolve_mouth_path(row):
    original_path = Path(str(row["mouth_path"]))

    if original_path.exists():
        return str(original_path)

    label_folder = str(row["label"]).strip().lower()
    reconstructed_path = (
        DATA_ROOT
        / label_folder
        / original_path.name
    )

    return str(reconstructed_path)


primary_candidates["resolved_mouth_path"] = (
    primary_candidates.apply(resolve_mouth_path, axis=1)
)

primary_candidates["mouth_file_exists"] = (
    primary_candidates["resolved_mouth_path"]
    .map(lambda value: Path(value).exists())
)

missing_successful_files = primary_candidates[
    ~primary_candidates["mouth_file_exists"]
]

if not missing_successful_files.empty:
    print(
        "UYARI: Bulunamayan başarılı ağız dosyası:",
        len(missing_successful_files),
    )

    display(
        missing_successful_files[
            [
                "file_name",
                "sample_id",
                "mouth_path",
                "resolved_mouth_path",
            ]
        ].head(10)
    )

# Öncelik sırası:
# 1) Dosyası gerçekten bulunan kayıt
# 2) En büyük ağız kutusu
# 3) En küçük face_id
primary_candidates = primary_candidates.sort_values(
    by=[
        "file_name",
        "mouth_file_exists",
        "mouth_area_px",
        "face_id",
    ],
    ascending=[
        True,
        False,
        False,
        True,
    ],
)

# Her seçilmiş frame için yalnızca bir ağız ROI bırak
primary_mouth = (
    primary_candidates
    .drop_duplicates(
        subset="file_name",
        keep="first",
    )
    .copy()
)

# Ortak seçim metadata'sını temel tablo yap
selection_clean = selection_check.copy()

selection_clean["official_label"] = (
    selection_clean["sinif"]
    .astype(str)
    .str.strip()
    .str.lower()
)

selection_clean["official_split"] = (
    selection_clean["split"]
    .astype(str)
    .str.strip()
    .str.lower()
)

# 3.000 seçilmiş frame ile birebir birleştir
paired_metadata = selection_clean.merge(
    primary_mouth,
    on="file_name",
    how="left",
    validate="one_to_one",
    suffixes=("_selection", "_mouth"),
)

paired_metadata["mouth_available"] = (
    paired_metadata["resolved_mouth_path"]
    .notna()
    & paired_metadata["mouth_file_exists"].fillna(False)
)

# Etiket ve split tutarlılık kontrolü
successful_paired = paired_metadata[
    paired_metadata["mouth_available"]
].copy()

label_mismatch = successful_paired[
    successful_paired["official_label"]
    != successful_paired["label"].astype(str).str.lower()
]

split_mismatch = successful_paired[
    successful_paired["official_split"]
    != successful_paired["split_mouth"].astype(str).str.lower()
]

if not label_mismatch.empty:
    raise AssertionError(
        f"{len(label_mismatch)} adet etiket uyuşmazlığı bulundu."
    )

if not split_mismatch.empty:
    raise AssertionError(
        f"{len(split_mismatch)} adet split uyuşmazlığı bulundu."
    )

print("ANA AĞIZ ROI SEÇİM SONUCU")
print("-" * 60)
print("Başlangıçtaki seçilmiş frame:", len(paired_metadata))
print("Kullanılabilir tekil ağız ROI:", len(successful_paired))
print(
    "Ağız bulunamadığı için kullanılamayan:",
    (~paired_metadata["mouth_available"]).sum(),
)
print(
    "Her frame için ağız sayısı en fazla 1:",
    successful_paired["file_name"].is_unique,
)
print(
    "Bütün seçilen ağız dosyaları mevcut:",
    successful_paired["mouth_file_exists"].all(),
)

print("\nSplit dağılımı:")
print(
    successful_paired["official_split"]
    .value_counts()
    .reindex(["train", "val", "test"])
)

print("\nEtiket dağılımı:")
print(
    successful_paired["official_label"]
    .value_counts()
    .reindex(["real", "fake"])
)

print("\nSeçilen ilk 5 ağız ROI:")
display(
    successful_paired[
        [
            "file_name",
            "official_label",
            "official_split",
            "face_id",
            "faces_in_frame",
            "mouth_area_px",
            "resolved_mouth_path",
        ]
    ].head()
)

UYARI: Bulunamayan başarılı ağız dosyası: 2987


,file_name,sample_id,mouth_path,resolved_mouth_path
0,fake_train_01130.jpg,fake_train_01130_face00,/content/drive/MyDrive/AISC Çalışmalar/Deney...,/content/drive/MyDrive/AISC DeepFake Çalışma...
1,fake_train_01130.jpg,fake_train_fake_train_01130_face00,/content/drive/MyDrive/AISC Çalışmalar/Deney...,/content/drive/MyDrive/AISC DeepFake Çalışma...
2,fake_train_00000.jpg,fake_train_fake_train_00000_face00,/content/drive/MyDrive/AISC Çalışmalar/Deney...,/content/drive/MyDrive/AISC DeepFake Çalışma...
3,fake_train_00001.jpg,fake_train_fake_train_00001_face00,/content/drive/MyDrive/AISC Çalışmalar/Deney...,/content/drive/MyDrive/AISC DeepFake Çalışma...
4,fake_train_00002.jpg,fake_train_fake_train_00002_face00,/content/drive/MyDrive/AISC Çalışmalar/Deney...,/content/drive/MyDrive/AISC DeepFake Çalışma...
5,fake_train_00003.jpg,fake_train_fake_train_00003_face00,/content/drive/MyDrive/AISC Çalışmalar/Deney...,/content/drive/MyDrive/AISC DeepFake Çalışma...
6,fake_train_00004.jpg,fake_train_fake_train_00004_face00,/content/drive/MyDrive/AISC Çalışmalar/Deney...,/content/drive/MyDrive/AISC DeepFake Çalışma...
7,fake_train_00005.jpg,fake_train_fake_train_00005_face00,/content/drive/MyDrive/AISC Çalışmalar/Deney...,/content/drive/MyDrive/AISC DeepFake Çalışma...
8,fake_train_00006.jpg,fake_train_fake_train_00006_face00,/content/drive/MyDrive/AISC Çalışmalar/Deney...,/content/drive/MyDrive/AISC DeepFake Çalışma...
9,fake_train_00007.jpg,fake_train_fake_train_00007_face00,/content/drive/MyDrive/AISC Çalışmalar/Deney...,/content/drive/MyDrive/AISC DeepFake Çalışma...


ANA AĞIZ ROI SEÇİM SONUCU
------------------------------------------------------------
Başlangıçtaki seçilmiş frame: 3000
Kullanılabilir tekil ağız ROI: 0
Ağız bulunamadığı için kullanılamayan: 3000
Her frame için ağız sayısı en fazla 1: True
Bütün seçilen ağız dosyaları mevcut: True

Split dağılımı:
official_split
train   NaN
val     NaN
test    NaN
Name: count, dtype: float64

Etiket dağılımı:
official_label
real   NaN
fake   NaN
Name: count, dtype: float64

Seçilen ilk 5 ağız ROI:


/tmp/ipykernel_1117/1938922272.py:143: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  & paired_metadata["mouth_file_exists"].fillna(False)


,file_name,official_label,official_split,face_id,faces_in_frame,mouth_area_px,resolved_mouth_path


In [8]:
# 7. HÜCRE — DÜZELTİLMİŞ ana ağız ROI seçimi

import pandas as pd
from pathlib import Path

primary_candidates = roi_check[
    roi_check["mouth_detected_bool"]
].copy()

# Sayısal sütunları dönüştür
for column in [
    "face_id",
    "mouth_width_px",
    "mouth_height_px",
    "mouth_bbox_x1",
    "mouth_bbox_y1",
    "mouth_bbox_x2",
    "mouth_bbox_y2",
]:
    primary_candidates[column] = pd.to_numeric(
        primary_candidates[column],
        errors="coerce",
    )

primary_candidates["mouth_area_px"] = (
    primary_candidates["mouth_width_px"]
    * primary_candidates["mouth_height_px"]
)

if primary_candidates["mouth_area_px"].isna().any():
    raise ValueError(
        "Ağız alanı hesaplanamayan başarılı kayıtlar bulundu."
    )


def resolve_mouth_path_safely(saved_path):
    """
    Metadata içindeki eski Drive kökünü kaldırır ve
    mouth_roi_output sonrasındaki alt klasör yapısını korur.
    """

    original_path = Path(str(saved_path))

    # Eski mutlak yol hâlâ çalışıyorsa doğrudan kullan
    if original_path.exists():
        return str(original_path)

    parts = list(original_path.parts)

    # mouth_roi_output sonrasındaki yolu al
    if "mouth_roi_output" in parts:
        marker_index = parts.index("mouth_roi_output")
        relative_parts = parts[marker_index + 1:]

        reconstructed_path = DATA_ROOT.joinpath(*relative_parts)

        if reconstructed_path.exists():
            return str(reconstructed_path)

    # Son güvenli seçenek: Aynı dosya adını klasör içinde ara
    matching_paths = list(DATA_ROOT.rglob(original_path.name))

    # Debug görüntülerini adaylardan çıkar
    non_debug_matches = [
        path
        for path in matching_paths
        if "debug" not in {
            part.lower()
            for part in path.parts
        }
    ]

    if len(non_debug_matches) == 1:
        return str(non_debug_matches[0])

    if len(matching_paths) == 1:
        return str(matching_paths[0])

    return None


primary_candidates["resolved_mouth_path"] = (
    primary_candidates["mouth_path"]
    .map(resolve_mouth_path_safely)
)

primary_candidates["mouth_file_exists"] = (
    primary_candidates["resolved_mouth_path"]
    .map(
        lambda value: (
            value is not None
            and Path(value).exists()
        )
    )
)

print(
    "Bulunan ağız ROI dosyası:",
    primary_candidates["mouth_file_exists"].sum(),
)

print(
    "Bulunamayan ağız ROI dosyası:",
    (~primary_candidates["mouth_file_exists"]).sum(),
)

# Yalnızca fiziksel dosyası bulunan kayıtlar
valid_candidates = primary_candidates[
    primary_candidates["mouth_file_exists"]
].copy()

# Her frame için:
# 1. En büyük ağız alanı
# 2. Eşitlikte en küçük face_id
valid_candidates = valid_candidates.sort_values(
    by=[
        "file_name",
        "mouth_area_px",
        "face_id",
    ],
    ascending=[
        True,
        False,
        True,
    ],
)

primary_mouth = (
    valid_candidates
    .drop_duplicates(
        subset="file_name",
        keep="first",
    )
    .copy()
)

# Resmî 3.000 frame listesini temel al
selection_clean = selection_check.copy()

selection_clean["official_label"] = (
    selection_clean["sinif"]
    .astype(str)
    .str.strip()
    .str.lower()
)

selection_clean["official_split"] = (
    selection_clean["split"]
    .astype(str)
    .str.strip()
    .str.lower()
)

paired_metadata = selection_clean.merge(
    primary_mouth,
    on="file_name",
    how="left",
    validate="one_to_one",
    suffixes=("_selection", "_mouth"),
)

paired_metadata["mouth_available"] = (
    paired_metadata["resolved_mouth_path"].notna()
)

successful_paired = paired_metadata[
    paired_metadata["mouth_available"]
].copy()

# Etiket ve split kontrolü
label_mismatch = successful_paired[
    successful_paired["official_label"]
    != successful_paired["label"].astype(str).str.lower()
]

split_mismatch = successful_paired[
    successful_paired["official_split"]
    != successful_paired["split_mouth"].astype(str).str.lower()
]

if not label_mismatch.empty:
    raise AssertionError(
        f"{len(label_mismatch)} etiket uyuşmazlığı bulundu."
    )

if not split_mismatch.empty:
    raise AssertionError(
        f"{len(split_mismatch)} split uyuşmazlığı bulundu."
    )

print("\nDÜZELTİLMİŞ ANA AĞIZ ROI SONUCU")
print("-" * 60)
print("Başlangıçtaki seçilmiş frame:", len(paired_metadata))
print("Kullanılabilir tekil ağız ROI:", len(successful_paired))
print(
    "Ağız bulunamadığı için kullanılamayan:",
    len(paired_metadata) - len(successful_paired),
)
print(
    "Her frame yalnızca bir kez bulunuyor:",
    successful_paired["file_name"].is_unique,
)
print(
    "Bütün seçilen dosyalar mevcut:",
    successful_paired["resolved_mouth_path"]
    .map(lambda value: Path(value).exists())
    .all(),
)

print("\nSplit dağılımı:")
print(
    successful_paired["official_split"]
    .value_counts()
    .reindex(["train", "val", "test"])
)

print("\nEtiket dağılımı:")
print(
    successful_paired["official_label"]
    .value_counts()
    .reindex(["real", "fake"])
)

display(
    successful_paired[
        [
            "file_name",
            "official_label",
            "official_split",
            "face_id",
            "faces_in_frame",
            "mouth_area_px",
            "resolved_mouth_path",
        ]
    ].head()
)

Bulunan ağız ROI dosyası: 2987
Bulunamayan ağız ROI dosyası: 0

DÜZELTİLMİŞ ANA AĞIZ ROI SONUCU
------------------------------------------------------------
Başlangıçtaki seçilmiş frame: 3000
Kullanılabilir tekil ağız ROI: 2889
Ağız bulunamadığı için kullanılamayan: 111
Her frame yalnızca bir kez bulunuyor: True
Bütün seçilen dosyalar mevcut: True

Split dağılımı:
official_split
train    2310
val       287
test      292
Name: count, dtype: int64

Etiket dağılımı:
official_label
real    1467
fake    1422
Name: count, dtype: int64


,file_name,official_label,official_split,face_id,faces_in_frame,mouth_area_px,resolved_mouth_path
0,real_train_00000.jpg,real,train,0.0,1.0,18245.0,/content/drive/MyDrive/AISC DeepFake Çalışma...
1,real_train_00001.jpg,real,train,0.0,1.0,1450.0,/content/drive/MyDrive/AISC DeepFake Çalışma...
2,real_train_00002.jpg,real,train,0.0,1.0,16799.0,/content/drive/MyDrive/AISC DeepFake Çalışma...
4,real_train_00004.jpg,real,train,0.0,1.0,3526.0,/content/drive/MyDrive/AISC DeepFake Çalışma...
5,real_train_00005.jpg,real,train,0.0,1.0,5474.0,/content/drive/MyDrive/AISC DeepFake Çalışma...


In [9]:
# 8. HÜCRE — Source-video split izolasyonu kontrolü

from pathlib import Path
import pandas as pd

isolation_metadata = successful_paired.copy()

# Orijinal frame yolundan kaynak video kimliğini çıkar
isolation_metadata["source_video"] = (
    isolation_metadata["orijinal_yol"]
    .astype(str)
    .map(lambda value: Path(value).parent.name)
)

# Boş kaynak video kimliği kontrolü
invalid_source_video = (
    isolation_metadata["source_video"]
    .isna()
    | isolation_metadata["source_video"].astype(str).str.strip().eq("")
)

if invalid_source_video.any():
    raise ValueError(
        f"{invalid_source_video.sum()} satırda kaynak video kimliği bulunamadı."
    )

split_video_ids = {
    split_name: set(
        isolation_metadata.loc[
            isolation_metadata["official_split"] == split_name,
            "source_video",
        ]
    )
    for split_name in ["train", "val", "test"]
}

train_val_overlap = (
    split_video_ids["train"]
    & split_video_ids["val"]
)

train_test_overlap = (
    split_video_ids["train"]
    & split_video_ids["test"]
)

val_test_overlap = (
    split_video_ids["val"]
    & split_video_ids["test"]
)

print("SOURCE-VIDEO DAĞILIMI")
print("-" * 60)
print("Train kaynak video:", len(split_video_ids["train"]))
print("Validation kaynak video:", len(split_video_ids["val"]))
print("Test kaynak video:", len(split_video_ids["test"]))

print("\nSOURCE-VIDEO KESİŞİMLERİ")
print("-" * 60)
print("Train–validation kesişimi:", len(train_val_overlap))
print("Train–test kesişimi:", len(train_test_overlap))
print("Validation–test kesişimi:", len(val_test_overlap))

if train_val_overlap:
    print("\nTrain–validation örnek kesişimleri:")
    print(sorted(train_val_overlap)[:10])

if train_test_overlap:
    print("\nTrain–test örnek kesişimleri:")
    print(sorted(train_test_overlap)[:10])

if val_test_overlap:
    print("\nValidation–test örnek kesişimleri:")
    print(sorted(val_test_overlap)[:10])

# Eğitim yalnızca bütün kesişimler sıfırsa devam eder
assert not train_val_overlap, (
    "Train ve validation arasında source-video leakage bulundu."
)

assert not train_test_overlap, (
    "Train ve test arasında source-video leakage bulundu."
)

assert not val_test_overlap, (
    "Validation ve test arasında source-video leakage bulundu."
)

print("\n✅ Source-video split izolasyonu başarıyla doğrulandı.")
print("✅ Aynı kaynak video farklı splitlerde bulunmuyor.")

print("\nİlk 5 kaynak video eşleşmesi:")
display(
    isolation_metadata[
        [
            "file_name",
            "official_label",
            "official_split",
            "source_video",
            "orijinal_yol",
        ]
    ].head()
)

SOURCE-VIDEO DAĞILIMI
------------------------------------------------------------
Train kaynak video: 1557
Validation kaynak video: 187
Test kaynak video: 196

SOURCE-VIDEO KESİŞİMLERİ
------------------------------------------------------------
Train–validation kesişimi: 0
Train–test kesişimi: 0
Validation–test kesişimi: 0

✅ Source-video split izolasyonu başarıyla doğrulandı.
✅ Aynı kaynak video farklı splitlerde bulunmuyor.

İlk 5 kaynak video eşleşmesi:


,file_name,official_label,official_split,source_video,orijinal_yol
0,real_train_00000.jpg,real,train,477,/content/drive/MyDrive/AISC DeepFake Çalışmala...
1,real_train_00001.jpg,real,train,471,/content/drive/MyDrive/AISC DeepFake Çalışmala...
2,real_train_00002.jpg,real,train,751,/content/drive/MyDrive/AISC DeepFake Çalışmala...
4,real_train_00004.jpg,real,train,413,/content/drive/MyDrive/AISC DeepFake Çalışmala...
5,real_train_00005.jpg,real,train,620,/content/drive/MyDrive/AISC DeepFake Çalışmala...


In [10]:
variables_to_check = [
    "DATA_ROOT",
    "roi_metadata",
    "selection_metadata",
    "successful_paired",
    "isolation_metadata",
]

for variable in variables_to_check:
    print(
        variable,
        "✅ mevcut" if variable in globals() else "❌ kayıp"
    )

DATA_ROOT ✅ mevcut
roi_metadata ✅ mevcut
selection_metadata ✅ mevcut
successful_paired ✅ mevcut
isolation_metadata ✅ mevcut


In [11]:
import torch

print("CUDA kullanılabilir:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU bağlı değil.")

CUDA kullanılabilir: True
GPU: Tesla T4


In [12]:
# 9. HÜCRE — SSOT uyumlu deney konfigürasyonu ve çıktı klasörleri

import os
import random
import unicodedata
from dataclasses import asdict, dataclass
from datetime import datetime
from pathlib import Path
from typing import Any, Optional, Tuple

import numpy as np
import torch
import yaml


@dataclass(frozen=True)
class ExperimentConfig:
    seed: int = 42
    region: str = "mouth"
    model_name: str = "vgg16scratch_hog_gist_rbf_svm"

    image_size: int = 128
    batch_size: int = 32
    num_workers: int = 2

    epochs: int = 30
    early_stopping_patience: int = 7
    learning_rate: float = 1e-4
    weight_decay: float = 1e-4
    dropout: float = 0.40
    classifier_hidden_dim: int = 256

    horizontal_flip_probability: float = 0.50
    affine_degrees: float = 5.0
    affine_translate: float = 0.03
    color_jitter_brightness: float = 0.08
    color_jitter_contrast: float = 0.08

    hog_orientations: int = 9
    hog_pixels_per_cell: Tuple[int, int] = (8, 8)
    hog_cells_per_block: Tuple[int, int] = (2, 2)

    gist_image_size: int = 128
    gist_grid_size: int = 4
    gist_orientations: int = 8
    gist_scales: Tuple[Tuple[float, float], ...] = (
        (2.0, 4.0),
        (3.0, 6.0),
        (4.0, 8.0),
        (5.0, 10.0),
    )

    pca_components: int = 256

    svm_c_values: Tuple[float, ...] = (
        0.1,
        1.0,
        10.0,
        100.0,
    )

    svm_gamma_values: Tuple[Any, ...] = (
        "scale",
        1e-3,
        1e-4,
    )

    svm_class_weight: str = "balanced"

    checkpoint_keep_last_n_epochs: int = 3

    # Kullanıcının rapor standardı
    figure_dpi: int = 600
    minimum_figure_short_edge_px: int = 600

    resume_run_id: Optional[str] = None


CONFIG = ExperimentConfig()


def set_global_seed(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def normalized_name(value: str) -> str:
    return unicodedata.normalize("NFC", str(value))


def find_or_create_child(
    parent: Path,
    expected_name: str,
) -> Path:
    expected_normalized = normalized_name(expected_name)

    if parent.exists():
        for child in parent.iterdir():
            if (
                child.is_dir()
                and normalized_name(child.name)
                == expected_normalized
            ):
                return child

    new_child = parent / expected_name
    new_child.mkdir(parents=True, exist_ok=True)
    return new_child


def atomic_write_text(
    content: str,
    target: Path,
) -> None:
    temporary_path = target.with_suffix(
        target.suffix + ".tmp"
    )

    temporary_path.write_text(
        content,
        encoding="utf-8",
    )

    if not temporary_path.exists():
        raise RuntimeError(
            f"Geçici dosya oluşturulamadı: {temporary_path}"
        )

    os.replace(temporary_path, target)


set_global_seed(CONFIG.seed)

# mouth_roi_output → Ağız → Deney 1
EXPERIMENT_ROOT = DATA_ROOT.parents[1]

# Var olan Sonuçlar klasörünü Unicode farklarına rağmen bul
RESULTS_ROOT = find_or_create_child(
    EXPERIMENT_ROOT,
    "Sonuçlar",
)

MODEL_RESULTS_ROOT = find_or_create_child(
    RESULTS_ROOT,
    "VGG16_HOG_GIST_RBF_SVM_Mouth",
)

timestamp = datetime.now().strftime("%Y%m%d_%H%M")

RUN_ID = (
    f"{timestamp}_"
    f"{CONFIG.region}_"
    f"{CONFIG.model_name}_"
    f"seed{CONFIG.seed}"
)

RUN_DIR = MODEL_RESULTS_ROOT / RUN_ID

if RUN_DIR.exists():
    raise FileExistsError(
        f"Aynı run klasörü zaten mevcut: {RUN_DIR}"
    )

OUTPUT_DIRS = {
    "checkpoints": RUN_DIR / "checkpoints",
    "logs": RUN_DIR / "logs",
    "metrics": RUN_DIR / "metrics",
    "predictions": RUN_DIR / "predictions",
    "figures": RUN_DIR / "figures",
    "artifacts": RUN_DIR / "artifacts",
}

for directory in [
    RUN_DIR,
    *OUTPUT_DIRS.values(),
]:
    directory.mkdir(
        parents=True,
        exist_ok=False,
    )

resolved_config = {
    **asdict(CONFIG),
    "run_id": RUN_ID,
    "data_root": str(DATA_ROOT),
    "roi_metadata_path": str(ROI_METADATA_PATH),
    "selection_metadata_path": str(
        SELECTION_METADATA_PATH
    ),
    "results_root": str(RESULTS_ROOT),
    "run_directory": str(RUN_DIR),
    "eligible_sample_count": len(successful_paired),
    "train_sample_count": int(
        (
            successful_paired["official_split"]
            == "train"
        ).sum()
    ),
    "validation_sample_count": int(
        (
            successful_paired["official_split"]
            == "val"
        ).sum()
    ),
    "test_sample_count": int(
        (
            successful_paired["official_split"]
            == "test"
        ).sum()
    ),
    "source_video_overlap": {
        "train_validation": len(train_val_overlap),
        "train_test": len(train_test_overlap),
        "validation_test": len(val_test_overlap),
    },
}

config_yaml = yaml.safe_dump(
    resolved_config,
    allow_unicode=True,
    sort_keys=False,
)

atomic_write_text(
    config_yaml,
    RUN_DIR / "config_resolved.yaml",
)

print("DENEY KLASÖRÜ HAZIR")
print("-" * 60)
print("Run ID:", RUN_ID)
print("Sonuç ana klasörü:", RESULTS_ROOT)
print("Bu deneyin klasörü:", RUN_DIR)

print("\nOluşturulan alt klasörler:")
for name, directory in OUTPUT_DIRS.items():
    print(f"{name}: {directory}")

print("\nConfig kaydedildi:")
print(RUN_DIR / "config_resolved.yaml")

print("\n✅ Ham ağız verilerine yazılmadı.")
print("✅ Yeni ve bağımsız run klasörü oluşturuldu.")

DENEY KLASÖRÜ HAZIR
------------------------------------------------------------
Run ID: 20260807_1410_mouth_vgg16scratch_hog_gist_rbf_svm_seed42
Sonuç ana klasörü: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Dilara/Deney 1/Sonuçlar
Bu deneyin klasörü: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Dilara/Deney 1/Sonuçlar/VGG16_HOG_GIST_RBF_SVM_Mouth/20260807_1410_mouth_vgg16scratch_hog_gist_rbf_svm_seed42

Oluşturulan alt klasörler:
checkpoints: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Dilara/Deney 1/Sonuçlar/VGG16_HOG_GIST_RBF_SVM_Mouth/20260807_1410_mouth_vgg16scratch_hog_gist_rbf_svm_seed42/checkpoints
logs: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Dilara/Deney 1/Sonuçlar/VGG16_HOG_GIST_RBF_SVM_Mouth/20260807_1410_mouth_vgg16scratch_hog_gist_rbf_svm_seed42/logs
metrics: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Dilara/Deney 1/Sonuçlar/VGG16_HOG_GIST_RBF_SVM_Mouth/20260807_1410_mouth_vgg16scrat

In [13]:
# 10. HÜCRE — Standart metadata, SHA-256 ve veri muhasebesi

import hashlib
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm


def calculate_sha256(file_path: Path) -> str:
    sha256_hash = hashlib.sha256()

    with file_path.open("rb") as file_handle:
        for block in iter(
            lambda: file_handle.read(1024 * 1024),
            b"",
        ):
            sha256_hash.update(block)

    return sha256_hash.hexdigest()


def extract_frame_index(path_value: str) -> int:
    stem = Path(str(path_value)).stem
    match = re.search(r"(\d+)$", stem)

    if match is None:
        return -1

    return int(match.group(1))


def create_stable_sample_id(row: pd.Series) -> str:
    stable_text = (
        f"{row['source_video']}|"
        f"{row['frame_index']}|"
        f"{row['face_index']}|"
        f"{row['label']}|"
        f"{row['file_name']}"
    )

    return hashlib.sha256(
        stable_text.encode("utf-8")
    ).hexdigest()[:16]


def atomic_write_csv(
    dataframe: pd.DataFrame,
    target: Path,
) -> None:
    temporary_path = target.with_suffix(
        target.suffix + ".tmp"
    )

    dataframe.to_csv(
        temporary_path,
        index=False,
    )

    if not temporary_path.exists():
        raise RuntimeError(
            f"Geçici CSV oluşturulamadı: {temporary_path}"
        )

    # Yazılan dosyanın yeniden okunabildiğini doğrula
    validation_frame = pd.read_csv(temporary_path)

    if len(validation_frame) != len(dataframe):
        raise RuntimeError(
            f"CSV satır doğrulaması başarısız: {target}"
        )

    os.replace(temporary_path, target)


standard_metadata = paired_metadata.copy()

standard_metadata["source_video"] = (
    standard_metadata["orijinal_yol"]
    .astype(str)
    .map(lambda value: Path(value).parent.name)
)

standard_metadata["frame_index"] = (
    standard_metadata["orijinal_yol"]
    .astype(str)
    .map(extract_frame_index)
)

standard_metadata["face_index"] = (
    pd.to_numeric(
        standard_metadata["face_id"],
        errors="coerce",
    )
    .fillna(-1)
    .astype(int)
)

standard_metadata["roi_state"] = "not_available"
standard_metadata["label"] = (
    standard_metadata["official_label"]
    .astype(str)
    .str.lower()
)

standard_metadata["split"] = (
    standard_metadata["official_split"]
    .astype(str)
    .str.lower()
)

standard_metadata["status"] = np.where(
    standard_metadata["mouth_available"],
    "SUCCESS",
    "SKIPPED",
)

standard_metadata["skip_reason"] = np.where(
    standard_metadata["mouth_available"],
    "",
    "MOUTH_NOT_DETECTED",
)

standard_metadata["output_path"] = (
    standard_metadata["resolved_mouth_path"]
    .fillna("")
    .astype(str)
)

standard_metadata["run_id"] = RUN_ID

# Başarılı örneklerin hash değerlerini hesapla
successful_paths = (
    standard_metadata.loc[
        standard_metadata["status"] == "SUCCESS",
        "output_path",
    ]
    .drop_duplicates()
    .tolist()
)

path_to_sha256 = {}

for path_value in tqdm(
    successful_paths,
    desc="Calculating SHA-256",
):
    image_path = Path(path_value)

    if not image_path.exists():
        raise FileNotFoundError(
            f"SUCCESS dosyası bulunamadı: {image_path}"
        )

    path_to_sha256[path_value] = calculate_sha256(
        image_path
    )

standard_metadata["sha256"] = (
    standard_metadata["output_path"]
    .map(path_to_sha256)
    .fillna("")
)

standard_metadata["sample_id"] = (
    standard_metadata.apply(
        create_stable_sample_id,
        axis=1,
    )
)

standard_columns = [
    "sample_id",
    "source_video",
    "frame_index",
    "face_index",
    "roi_state",
    "label",
    "split",
    "status",
    "skip_reason",
    "sha256",
    "output_path",
    "run_id",
    "file_name",
    "orijinal_yol",
    "mouth_area_px",
    "faces_in_frame",
]

metadata_full_audit = standard_metadata[
    standard_columns
].copy()

metadata_used = metadata_full_audit[
    metadata_full_audit["status"] == "SUCCESS"
].copy()

metadata_used["target"] = (
    metadata_used["label"]
    .map(
        {
            "real": 0,
            "fake": 1,
        }
    )
    .astype(int)
)

# Veri muhasebesi
total_inputs = len(metadata_full_audit)

success_count = int(
    (metadata_full_audit["status"] == "SUCCESS").sum()
)

skipped_count = int(
    (metadata_full_audit["status"] == "SKIPPED").sum()
)

error_count = int(
    (metadata_full_audit["status"] == "ERROR").sum()
)

assert (
    total_inputs
    == success_count + skipped_count + error_count
), "Girdi-çıktı veri muhasebesi uyuşmuyor."

assert metadata_full_audit["sample_id"].is_unique, (
    "Mükerrer sample_id bulundu."
)

assert metadata_used["file_name"].is_unique, (
    "Eğitim metadata'sında mükerrer frame bulundu."
)

assert metadata_used["output_path"].map(
    lambda value: Path(value).exists()
).all(), "SUCCESS durumundaki bazı dosyalar bulunamadı."

assert metadata_used["sha256"].str.len().eq(64).all(), (
    "Geçersiz SHA-256 değeri bulundu."
)

# Hash değerlerinin splitler arasındaki kesişimi
hashes_by_split = {
    split_name: set(
        metadata_used.loc[
            metadata_used["split"] == split_name,
            "sha256",
        ]
    )
    for split_name in ["train", "val", "test"]
}

train_val_hash_overlap = (
    hashes_by_split["train"]
    & hashes_by_split["val"]
)

train_test_hash_overlap = (
    hashes_by_split["train"]
    & hashes_by_split["test"]
)

val_test_hash_overlap = (
    hashes_by_split["val"]
    & hashes_by_split["test"]
)

print("\nVERİ MUHASEBESİ")
print("-" * 60)
print("Toplam girdi:", total_inputs)
print("SUCCESS:", success_count)
print("SKIPPED:", skipped_count)
print("ERROR:", error_count)

print("\nHASH SPLIT KESİŞİMLERİ")
print("-" * 60)
print(
    "Train-validation:",
    len(train_val_hash_overlap),
)
print(
    "Train-test:",
    len(train_test_hash_overlap),
)
print(
    "Validation-test:",
    len(val_test_hash_overlap),
)

assert not train_val_hash_overlap, (
    "Train ve validation arasında aynı görüntü içeriği bulundu."
)

assert not train_test_hash_overlap, (
    "Train ve test arasında aynı görüntü içeriği bulundu."
)

assert not val_test_hash_overlap, (
    "Validation ve test arasında aynı görüntü içeriği bulundu."
)

full_audit_path = (
    OUTPUT_DIRS["artifacts"]
    / "metadata_full_audit.csv"
)

metadata_used_path = (
    OUTPUT_DIRS["artifacts"]
    / "metadata_used.csv"
)

atomic_write_csv(
    metadata_full_audit,
    full_audit_path,
)

atomic_write_csv(
    metadata_used,
    metadata_used_path,
)

print("\nKAYDEDİLEN DOSYALAR")
print("-" * 60)
print("Tam audit metadata:", full_audit_path)
print("Eğitim metadata:", metadata_used_path)

print("\n✅ Veri muhasebesi doğrulandı.")
print("✅ Source-video izolasyonu korunuyor.")
print("✅ Splitler arasında aynı görüntü hash'i yok.")
print("✅ Orijinal metadata değiştirilmedi.")

Calculating SHA-256:   0%|          | 0/2889 [00:00<?, ?it/s]


VERİ MUHASEBESİ
------------------------------------------------------------
Toplam girdi: 3000
SUCCESS: 2889
SKIPPED: 111
ERROR: 0

HASH SPLIT KESİŞİMLERİ
------------------------------------------------------------
Train-validation: 0
Train-test: 0
Validation-test: 0

KAYDEDİLEN DOSYALAR
------------------------------------------------------------
Tam audit metadata: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Dilara/Deney 1/Sonuçlar/VGG16_HOG_GIST_RBF_SVM_Mouth/20260807_1410_mouth_vgg16scratch_hog_gist_rbf_svm_seed42/artifacts/metadata_full_audit.csv
Eğitim metadata: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Dilara/Deney 1/Sonuçlar/VGG16_HOG_GIST_RBF_SVM_Mouth/20260807_1410_mouth_vgg16scratch_hog_gist_rbf_svm_seed42/artifacts/metadata_used.csv

✅ Veri muhasebesi doğrulandı.
✅ Source-video izolasyonu korunuyor.
✅ Splitler arasında aynı görüntü hash'i yok.
✅ Orijinal metadata değiştirilmedi.


In [14]:
print(
    "metadata_used:",
    "✅ mevcut" if "metadata_used" in globals() else "❌ kayıp",
)

print(
    "RUN_DIR:",
    "✅ mevcut" if "RUN_DIR" in globals() else "❌ kayıp",
)

print(
    "OUTPUT_DIRS:",
    "✅ mevcut" if "OUTPUT_DIRS" in globals() else "❌ kayıp",
)

metadata_used: ✅ mevcut
RUN_DIR: ✅ mevcut
OUTPUT_DIRS: ✅ mevcut


In [15]:
# 11. HÜCRE — Veri dağılımı denetimi ve 600 DPI grafik

import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image


# Split tablolarını oluştur
train_frame = metadata_used[
    metadata_used["split"] == "train"
].copy()

val_frame = metadata_used[
    metadata_used["split"] == "val"
].copy()

test_frame = metadata_used[
    metadata_used["split"] == "test"
].copy()

assert len(train_frame) > 0, "Train kümesi boş."
assert len(val_frame) > 0, "Validation kümesi boş."
assert len(test_frame) > 0, "Test kümesi boş."

assert set(metadata_used["label"]) == {
    "real",
    "fake",
}, "Beklenen real/fake etiketleri bulunamadı."

assert set(metadata_used["split"]) == {
    "train",
    "val",
    "test",
}, "Beklenen train/val/test splitleri bulunamadı."


distribution_table = (
    metadata_used
    .groupby(
        ["split", "label"],
        observed=True,
    )
    .size()
    .unstack(fill_value=0)
    .reindex(
        index=["train", "val", "test"],
        columns=["real", "fake"],
        fill_value=0,
    )
)

distribution_table.index = [
    "Train",
    "Validation",
    "Test",
]

distribution_table.columns = [
    "Real",
    "Fake",
]

print("DATASET DISTRIBUTION")
print("-" * 60)
display(distribution_table)

print("\nTOTALS")
print("-" * 60)
print("Train:", len(train_frame))
print("Validation:", len(val_frame))
print("Test:", len(test_frame))
print("Total:", len(metadata_used))


distribution_csv_path = (
    OUTPUT_DIRS["metrics"]
    / "dataset_distribution.csv"
)

atomic_write_csv(
    distribution_table.reset_index(
        names="Split"
    ),
    distribution_csv_path,
)


# İngilizce ve standart renkli grafik
figure_path = (
    OUTPUT_DIRS["figures"]
    / "dataset_distribution.png"
)

temporary_figure_path = (
    OUTPUT_DIRS["figures"]
    / "dataset_distribution.tmp.png"
)

fig, ax = plt.subplots(
    figsize=(10, 6),
    dpi=150,
)

x_positions = np.arange(
    len(distribution_table.index)
)

bar_width = 0.36

real_bars = ax.bar(
    x_positions - bar_width / 2,
    distribution_table["Real"],
    width=bar_width,
    label="Real",
    color="#3182CE",
    edgecolor="#1A365D",
    linewidth=0.8,
)

fake_bars = ax.bar(
    x_positions + bar_width / 2,
    distribution_table["Fake"],
    width=bar_width,
    label="Fake",
    color="#DD6B20",
    edgecolor="#7B341E",
    linewidth=0.8,
)

ax.set_title(
    "Distribution of Mouth ROI Samples by Split and Class",
    fontsize=14,
    fontweight="bold",
    pad=14,
)

ax.set_xlabel(
    "Dataset Split",
    fontsize=11,
)

ax.set_ylabel(
    "Number of Samples",
    fontsize=11,
)

ax.set_xticks(x_positions)
ax.set_xticklabels(
    distribution_table.index,
    fontsize=11,
)

ax.legend(
    title="Class",
    frameon=True,
    loc="upper right",
)

ax.grid(
    axis="y",
    alpha=0.25,
    linestyle="--",
)

ax.set_axisbelow(True)

ax.bar_label(
    real_bars,
    padding=4,
    fontsize=10,
)

ax.bar_label(
    fake_bars,
    padding=4,
    fontsize=10,
)

fig.tight_layout()

fig.savefig(
    temporary_figure_path,
    dpi=CONFIG.figure_dpi,
    bbox_inches="tight",
    format="png",
)

plt.close(fig)

if not temporary_figure_path.exists():
    raise RuntimeError(
        "Geçici veri dağılımı grafiği oluşturulamadı."
    )

# Görsel çözünürlüğünü doğrula
with Image.open(temporary_figure_path) as image:
    figure_width, figure_height = image.size

if min(figure_width, figure_height) < (
    CONFIG.minimum_figure_short_edge_px
):
    raise AssertionError(
        "Grafik çözünürlüğü yetersiz: "
        f"{figure_width}x{figure_height}"
    )

os.replace(
    temporary_figure_path,
    figure_path,
)

print("\nKAYDEDİLEN ÇIKTILAR")
print("-" * 60)
print("Dağılım CSV:", distribution_csv_path)
print("Dağılım grafiği:", figure_path)
print(
    "Grafik çözünürlüğü:",
    f"{figure_width}x{figure_height}",
)
print("Grafik DPI:", CONFIG.figure_dpi)

print("\n✅ Veri dağılımı doğrulandı.")
print("✅ Grafik dili tamamen İngilizce.")
print("✅ Grafik kalite standardını karşılıyor.")

DATASET DISTRIBUTION
------------------------------------------------------------


,Real,Fake
Train,1172,1138
Validation,150,137
Test,145,147



TOTALS
------------------------------------------------------------
Train: 2310
Validation: 287
Test: 292
Total: 2889

KAYDEDİLEN ÇIKTILAR
------------------------------------------------------------
Dağılım CSV: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Dilara/Deney 1/Sonuçlar/VGG16_HOG_GIST_RBF_SVM_Mouth/20260807_1410_mouth_vgg16scratch_hog_gist_rbf_svm_seed42/metrics/dataset_distribution.csv
Dağılım grafiği: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Dilara/Deney 1/Sonuçlar/VGG16_HOG_GIST_RBF_SVM_Mouth/20260807_1410_mouth_vgg16scratch_hog_gist_rbf_svm_seed42/figures/dataset_distribution.png
Grafik çözünürlüğü: 5938x3538
Grafik DPI: 600

✅ Veri dağılımı doğrulandı.
✅ Grafik dili tamamen İngilizce.
✅ Grafik kalite standardını karşılıyor.


In [16]:
# ============================================================
# HÜCRE 12 — TRAIN-ONLY RGB MEAN / STD HESAPLAMA
# ============================================================

import os
import json
import cv2
import numpy as np
from tqdm.auto import tqdm
from pathlib import Path

IMAGE_SIZE = int(getattr(CONFIG, "image_size", 128))


def letterbox_rgb(image_path, target_size=128):
    """
    Görüntüyü en-boy oranını bozmadan target_size x target_size
    boyutuna getirir. Boş alanlar siyah ile doldurulur.
    """
    image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)

    if image is None:
        raise ValueError(f"Görüntü okunamadı: {image_path}")

    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    height, width = image.shape[:2]
    scale = min(target_size / width, target_size / height)

    new_width = max(1, int(round(width * scale)))
    new_height = max(1, int(round(height * scale)))

    interpolation = (
        cv2.INTER_AREA
        if scale < 1
        else cv2.INTER_LINEAR
    )

    resized = cv2.resize(
        image,
        (new_width, new_height),
        interpolation=interpolation
    )

    canvas = np.zeros(
        (target_size, target_size, 3),
        dtype=np.uint8
    )

    x_start = (target_size - new_width) // 2
    y_start = (target_size - new_height) // 2

    canvas[
        y_start:y_start + new_height,
        x_start:x_start + new_width
    ] = resized

    return canvas


# Split değişkeni bağlantı sonrasında kaybolmuşsa yeniden oluştur
if "train_frame" not in globals():
    train_frame = (
        metadata_used[
            metadata_used["split"].astype(str).str.lower().eq("train")
        ]
        .copy()
        .reset_index(drop=True)
    )

print("Train görüntüsü sayısı:", len(train_frame))
print("Görüntü boyutu:", IMAGE_SIZE, "x", IMAGE_SIZE)

if len(train_frame) == 0:
    raise RuntimeError("Train verisi boş görünüyor.")

# Kanal bazında toplamlar
channel_sum = np.zeros(3, dtype=np.float64)
channel_squared_sum = np.zeros(3, dtype=np.float64)

successful_images = 0
failed_images = []

for row in tqdm(
    train_frame.itertuples(index=False),
    total=len(train_frame),
    desc="Calculating train RGB statistics"
):
    image_path = getattr(row, "output_path")

    try:
        image = letterbox_rgb(
            image_path,
            target_size=IMAGE_SIZE
        ).astype(np.float64) / 255.0

        pixels = image.reshape(-1, 3)

        channel_sum += pixels.sum(axis=0)
        channel_squared_sum += np.square(pixels).sum(axis=0)

        successful_images += 1

    except Exception as error:
        failed_images.append({
            "image_path": str(image_path),
            "error": str(error)
        })

if failed_images:
    raise RuntimeError(
        f"Mean/std hesabında {len(failed_images)} görüntü okunamadı. "
        f"İlk hata: {failed_images[0]}"
    )

total_pixel_count = (
    successful_images *
    IMAGE_SIZE *
    IMAGE_SIZE
)

rgb_mean = channel_sum / total_pixel_count

rgb_variance = (
    channel_squared_sum / total_pixel_count
) - np.square(rgb_mean)

rgb_variance = np.maximum(rgb_variance, 0.0)
rgb_std = np.sqrt(rgb_variance)

normalization_data = {
    "calculation_scope": "train_only",
    "color_space": "RGB",
    "pixel_range": "[0, 1]",
    "resize_method": "aspect_ratio_preserving_letterbox",
    "padding_color_rgb": [0, 0, 0],
    "image_size": [IMAGE_SIZE, IMAGE_SIZE],
    "train_image_count": int(successful_images),
    "total_pixel_count": int(total_pixel_count),
    "mean": [float(value) for value in rgb_mean],
    "std": [float(value) for value in rgb_std],
    "seed": int(getattr(CONFIG, "seed", 42)),
    "run_id": str(RUN_ID)
}

normalization_path = Path(
    OUTPUT_DIRS["artifacts"]
) / "train_normalization.json"

temporary_path = normalization_path.with_suffix(".json.tmp")

with open(temporary_path, "w", encoding="utf-8") as file:
    json.dump(
        normalization_data,
        file,
        ensure_ascii=False,
        indent=2
    )

os.replace(temporary_path, normalization_path)

# Sonraki hücrelerde kullanılacak değişkenler
TRAIN_RGB_MEAN = tuple(normalization_data["mean"])
TRAIN_RGB_STD = tuple(normalization_data["std"])

print("\nTRAIN-ONLY NORMALIZATION")
print("-" * 52)
print("Kullanılan train görüntüsü:", successful_images)
print("Okunamayan görüntü:", len(failed_images))
print("RGB mean:", np.round(TRAIN_RGB_MEAN, 6))
print("RGB std :", np.round(TRAIN_RGB_STD, 6))
print("\nKaydedildi:", normalization_path)

assert successful_images == len(train_frame)
assert np.all(np.asarray(TRAIN_RGB_STD) > 0)
assert normalization_path.exists()

print("\n✅ Mean/std yalnızca train splitinden hesaplandı.")
print("✅ Validation ve test hesaba katılmadı.")
print("✅ Normalizasyon değerleri deney klasörüne kaydedildi.")
print("✅ Ham ağız görüntüleri değiştirilmedi.")

Train görüntüsü sayısı: 2310
Görüntü boyutu: 128 x 128


Calculating train RGB statistics:   0%|          | 0/2310 [00:00<?, ?it/s]


TRAIN-ONLY NORMALIZATION
----------------------------------------------------
Kullanılan train görüntüsü: 2310
Okunamayan görüntü: 0
RGB mean: [0.331613 0.224498 0.200019]
RGB std : [0.33399  0.2372   0.215962]

Kaydedildi: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Dilara/Deney 1/Sonuçlar/VGG16_HOG_GIST_RBF_SVM_Mouth/20260807_1410_mouth_vgg16scratch_hog_gist_rbf_svm_seed42/artifacts/train_normalization.json

✅ Mean/std yalnızca train splitinden hesaplandı.
✅ Validation ve test hesaba katılmadı.
✅ Normalizasyon değerleri deney klasörüne kaydedildi.
✅ Ham ağız görüntüleri değiştirilmedi.


In [21]:
# ============================================================
# HÜCRE 13 — DATASET, AUGMENTATION VE DATALOADER
# ============================================================

import os
import json
import random
import numpy as np
import torch

from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms


# ------------------------------------------------------------
# 1. Ayarlar ve tekrarlanabilirlik
# ------------------------------------------------------------

SEED = int(getattr(CONFIG, "seed", 42))
BATCH_SIZE = int(getattr(CONFIG, "batch_size", 32))
NUM_WORKERS = int(getattr(CONFIG, "num_workers", 2))
IMAGE_SIZE = int(getattr(CONFIG, "image_size", 128))

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)


def seed_worker(worker_id):
    worker_seed = SEED + worker_id
    random.seed(worker_seed)
    np.random.seed(worker_seed)
    torch.manual_seed(worker_seed)


loader_generator = torch.Generator()
loader_generator.manual_seed(SEED)


# ------------------------------------------------------------
# 2. Split tabloları
# ------------------------------------------------------------

def create_split_frame(split_name):
    return (
        metadata_used[
            metadata_used["split"]
            .astype(str)
            .str.lower()
            .eq(split_name)
        ]
        .copy()
        .reset_index(drop=True)
    )


train_frame = create_split_frame("train")
val_frame = create_split_frame("val")
test_frame = create_split_frame("test")

assert len(train_frame) == 2310
assert len(val_frame) == 287
assert len(test_frame) == 292


# ------------------------------------------------------------
# 3. Transformlar
# ------------------------------------------------------------

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.50),

    transforms.RandomAffine(
        degrees=5,
        translate=(0.03, 0.03),
        fill=0
    ),

    transforms.ColorJitter(
        brightness=0.08,
        contrast=0.08,
        saturation=0.0,
        hue=0.0
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=TRAIN_RGB_MEAN,
        std=TRAIN_RGB_STD
    )
])


evaluation_transform = transforms.Compose([
    transforms.ToTensor(),

    transforms.Normalize(
        mean=TRAIN_RGB_MEAN,
        std=TRAIN_RGB_STD
    )
])


# ------------------------------------------------------------
# 4. Dataset sınıfı
# ------------------------------------------------------------

class MouthROIDataset(Dataset):

    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True).copy()
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        image_path = str(row["output_path"])

        image = letterbox_rgb(
            image_path=image_path,
            target_size=IMAGE_SIZE
        )

        image = Image.fromarray(image)

        if self.transform is not None:
            image = self.transform(image)

        label_text = str(row["label"]).strip().lower()

        if label_text == "real":
            target = 0
        elif label_text == "fake":
            target = 1
        else:
            raise ValueError(
                f"Bilinmeyen etiket: {row['label']} | "
                f"Dosya: {image_path}"
            )

        return {
            "image": image,
            "target": torch.tensor(target, dtype=torch.long),
            "index": torch.tensor(index, dtype=torch.long),
            "path": image_path,
            "file_name": str(row["file_name"]),
            "source_video": str(row["source_video"])
        }


# ------------------------------------------------------------
# 5. Dataset nesneleri
# ------------------------------------------------------------

train_dataset = MouthROIDataset(
    dataframe=train_frame,
    transform=train_transform
)

val_dataset = MouthROIDataset(
    dataframe=val_frame,
    transform=evaluation_transform
)

test_dataset = MouthROIDataset(
    dataframe=test_frame,
    transform=evaluation_transform
)


# ------------------------------------------------------------
# 6. DataLoader nesneleri
# ------------------------------------------------------------

common_loader_arguments = {
    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "pin_memory": torch.cuda.is_available(),
    "worker_init_fn": seed_worker,
    "generator": loader_generator,
    "persistent_workers": NUM_WORKERS > 0
}

train_loader = DataLoader(
    train_dataset,
    shuffle=True,
    drop_last=False,
    **common_loader_arguments
)

val_loader = DataLoader(
    val_dataset,
    shuffle=False,
    drop_last=False,
    **common_loader_arguments
)

test_loader = DataLoader(
    test_dataset,
    shuffle=False,
    drop_last=False,
    **common_loader_arguments
)


# ------------------------------------------------------------
# 7. İlk batch kontrolü
# ------------------------------------------------------------

first_train_batch = next(iter(train_loader))
first_val_batch = next(iter(val_loader))
first_test_batch = next(iter(test_loader))

expected_train_batch_size = min(BATCH_SIZE, len(train_dataset))

expected_shape = (
    expected_train_batch_size,
    3,
    IMAGE_SIZE,
    IMAGE_SIZE
)

print("DATASET AND DATALOADER CHECK")
print("-" * 54)

print("Train dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))
print("Test dataset:", len(test_dataset))

print(
    "\nİlk train batch görüntü şekli:",
    tuple(first_train_batch["image"].shape)
)

print(
    "İlk train batch etiket şekli:",
    tuple(first_train_batch["target"].shape)
)

print(
    "Train batch etiketleri:",
    first_train_batch["target"].tolist()
)

print(
    "\nValidation batch şekli:",
    tuple(first_val_batch["image"].shape)
)

print(
    "Test batch şekli:",
    tuple(first_test_batch["image"].shape)
)


# ------------------------------------------------------------
# 8. Kalite kontrolleri
# ------------------------------------------------------------

val_image_first = val_dataset[0]["image"]
val_image_second = val_dataset[0]["image"]

validation_is_deterministic = torch.equal(
    val_image_first,
    val_image_second
)

assert tuple(first_train_batch["image"].shape) == expected_shape
assert first_train_batch["image"].dtype == torch.float32
assert validation_is_deterministic

assert set(
    first_train_batch["target"].tolist()
).issubset({0, 1})


# ------------------------------------------------------------
# 9. Pipeline ayarlarını Drive'a kaydet
# ------------------------------------------------------------

data_pipeline_config = {
    "run_id": str(RUN_ID),
    "seed": SEED,
    "image_size": IMAGE_SIZE,
    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,

    "class_mapping": {
        "real": 0,
        "fake": 1
    },

    "resize_method": (
        "aspect_ratio_preserving_letterbox"
    ),

    "normalization": {
        "scope": "train_only",
        "mean_rgb": [
            float(value)
            for value in TRAIN_RGB_MEAN
        ],
        "std_rgb": [
            float(value)
            for value in TRAIN_RGB_STD
        ]
    },

    "train_augmentation": {
        "random_horizontal_flip_probability": 0.50,
        "random_affine_degrees": 5,
        "random_affine_translate": [0.03, 0.03],
        "brightness": 0.08,
        "contrast": 0.08
    },

    "validation_augmentation": None,
    "test_augmentation": None,

    "split_sizes": {
        "train": len(train_dataset),
        "validation": len(val_dataset),
        "test": len(test_dataset)
    }
}

pipeline_config_path = (
    Path(OUTPUT_DIRS["artifacts"])
    / "data_pipeline_config.json"
)

temporary_config_path = (
    pipeline_config_path.with_suffix(".json.tmp")
)

with open(
    temporary_config_path,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        data_pipeline_config,
        file,
        ensure_ascii=False,
        indent=2
    )

os.replace(
    temporary_config_path,
    pipeline_config_path
)

print("\nKaydedildi:", pipeline_config_path)

print("\n✅ Dataset ve DataLoader yapıları hazır.")
print("✅ Augmentation yalnızca train splitinde.")
print("✅ Validation ve test dönüşümleri deterministik.")
print("✅ Real=0, Fake=1 sınıf eşleştirmesi doğrulandı.")

DATASET AND DATALOADER CHECK
------------------------------------------------------
Train dataset: 2310
Validation dataset: 287
Test dataset: 292

İlk train batch görüntü şekli: (32, 3, 128, 128)
İlk train batch etiket şekli: (32,)
Train batch etiketleri: [1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1]

Validation batch şekli: (32, 3, 128, 128)
Test batch şekli: (32, 3, 128, 128)

Kaydedildi: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Dilara/Deney 1/Sonuçlar/VGG16_HOG_GIST_RBF_SVM_Mouth/20260807_1410_mouth_vgg16scratch_hog_gist_rbf_svm_seed42/artifacts/data_pipeline_config.json

✅ Dataset ve DataLoader yapıları hazır.
✅ Augmentation yalnızca train splitinde.
✅ Validation ve test dönüşümleri deterministik.
✅ Real=0, Fake=1 sınıf eşleştirmesi doğrulandı.


In [22]:
# ============================================================
# HÜCRE 14 — VGG16 SCRATCH + SMOKE TEST + CHECKPOINT TESTİ
# ============================================================

import os
import gc
import json
import random
import numpy as np
import torch
import torch.nn as nn

from pathlib import Path
from torchvision import models


# ------------------------------------------------------------
# 1. Cihaz ve hiperparametreler
# ------------------------------------------------------------

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

SEED = int(getattr(CONFIG, "seed", 42))
LEARNING_RATE = float(
    getattr(CONFIG, "learning_rate", 1e-4)
)
WEIGHT_DECAY = float(
    getattr(CONFIG, "weight_decay", 1e-4)
)
DROPOUT = float(getattr(CONFIG, "dropout", 0.40))
HIDDEN_DIM = int(getattr(CONFIG, "hidden_dim", 256))


def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)


set_all_seeds(SEED)

print("DEVICE CHECK")
print("-" * 54)
print("Kullanılan cihaz:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("UYARI: GPU bulunamadı.")


# ------------------------------------------------------------
# 2. VGG16 — sıfırdan eğitim
# ------------------------------------------------------------

class VGG16MouthClassifier(nn.Module):

    def __init__(
        self,
        hidden_dim=256,
        dropout=0.40,
        number_of_classes=2
    ):
        super().__init__()

        # weights=None:
        # ImageNet veya başka bir ön eğitim kullanılmaz.
        backbone = models.vgg16(weights=None)

        self.features = backbone.features

        # Her görüntüden 512 boyutlu CNN özellik vektörü
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout),
            nn.Linear(hidden_dim, number_of_classes)
        )

    def forward_features(self, images):
        features = self.features(images)
        features = self.avgpool(features)
        features = torch.flatten(features, 1)
        return features

    def forward(self, images):
        features = self.forward_features(images)
        logits = self.classifier(features)
        return logits


# ------------------------------------------------------------
# 3. Geçici smoke-test modeli
# ------------------------------------------------------------

smoke_model = VGG16MouthClassifier(
    hidden_dim=HIDDEN_DIM,
    dropout=DROPOUT,
    number_of_classes=2
).to(DEVICE)

smoke_criterion = nn.CrossEntropyLoss()

smoke_optimizer = torch.optim.AdamW(
    smoke_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

smoke_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    smoke_optimizer,
    mode="min",
    factor=0.5,
    patience=2
)


# ------------------------------------------------------------
# 4. Tek mini-batch ileri/geri yayılım testi
# ------------------------------------------------------------

smoke_images = first_train_batch["image"][:4].to(
    DEVICE,
    non_blocking=True
)

smoke_targets = first_train_batch["target"][:4].to(
    DEVICE,
    non_blocking=True
)

smoke_model.train()
smoke_optimizer.zero_grad(set_to_none=True)

smoke_logits = smoke_model(smoke_images)
smoke_loss = smoke_criterion(
    smoke_logits,
    smoke_targets
)

smoke_loss.backward()
smoke_optimizer.step()
smoke_scheduler.step(float(smoke_loss.item()))

assert smoke_logits.shape == (4, 2)
assert torch.isfinite(smoke_loss)
assert all(
    parameter.grad is None
    or torch.isfinite(parameter.grad).all()
    for parameter in smoke_model.parameters()
)

print("\nSMOKE TEST")
print("-" * 54)
print("Girdi şekli:", tuple(smoke_images.shape))
print("Çıktı şekli:", tuple(smoke_logits.shape))
print("Smoke loss:", round(smoke_loss.item(), 6))


# ------------------------------------------------------------
# 5. RNG durumunu kaydet
# ------------------------------------------------------------

rng_state = {
    "python": random.getstate(),
    "numpy": np.random.get_state(),
    "torch_cpu": torch.get_rng_state()
}

if torch.cuda.is_available():
    rng_state["torch_cuda"] = torch.cuda.get_rng_state_all()


# ------------------------------------------------------------
# 6. Atomik checkpoint kaydetme
# ------------------------------------------------------------

smoke_checkpoint_path = (
    Path(OUTPUT_DIRS["checkpoints"])
    / "smoke_test_checkpoint.pt"
)

temporary_checkpoint_path = (
    smoke_checkpoint_path.with_suffix(".pt.tmp")
)

smoke_checkpoint = {
    "checkpoint_type": "smoke_test",
    "run_id": str(RUN_ID),
    "seed": SEED,
    "epoch": 0,

    "model_name": "VGG16_from_scratch",
    "weights": None,

    "model_state_dict": smoke_model.state_dict(),
    "optimizer_state_dict": smoke_optimizer.state_dict(),
    "scheduler_state_dict": smoke_scheduler.state_dict(),

    "rng_state": rng_state,
    "smoke_loss": float(smoke_loss.item()),

    "class_mapping": {
        "real": 0,
        "fake": 1
    }
}

torch.save(
    smoke_checkpoint,
    temporary_checkpoint_path
)

os.replace(
    temporary_checkpoint_path,
    smoke_checkpoint_path
)


# ------------------------------------------------------------
# 7. Checkpoint yeniden yükleme testi
# ------------------------------------------------------------

loaded_smoke_checkpoint = torch.load(
    smoke_checkpoint_path,
    map_location=DEVICE,
    weights_only=False
)

required_checkpoint_keys = {
    "model_state_dict",
    "optimizer_state_dict",
    "scheduler_state_dict",
    "rng_state",
    "run_id",
    "seed"
}

missing_keys = (
    required_checkpoint_keys
    - set(loaded_smoke_checkpoint.keys())
)

assert len(missing_keys) == 0
assert loaded_smoke_checkpoint["run_id"] == str(RUN_ID)

verification_model = VGG16MouthClassifier(
    hidden_dim=HIDDEN_DIM,
    dropout=DROPOUT,
    number_of_classes=2
).to(DEVICE)

verification_model.load_state_dict(
    loaded_smoke_checkpoint["model_state_dict"]
)

verification_model.eval()

with torch.no_grad():
    verification_logits = verification_model(
        smoke_images
    )

assert verification_logits.shape == (4, 2)

print("\nCHECKPOINT TEST")
print("-" * 54)
print("Checkpoint:", smoke_checkpoint_path)
print("Eksik anahtar:", len(missing_keys))
print("Yeniden yükleme:", "Başarılı")


# ------------------------------------------------------------
# 8. Model mimarisi bilgisini kaydet
# ------------------------------------------------------------

total_parameters = sum(
    parameter.numel()
    for parameter in smoke_model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in smoke_model.parameters()
    if parameter.requires_grad
)

architecture_info = {
    "run_id": str(RUN_ID),
    "model": "VGG16",
    "initialization": "from_scratch",
    "torchvision_weights": None,
    "roi": "mouth",
    "input_shape": [3, IMAGE_SIZE, IMAGE_SIZE],
    "cnn_feature_dimension": 512,
    "classifier_hidden_dimension": HIDDEN_DIM,
    "dropout": DROPOUT,
    "number_of_classes": 2,
    "total_parameters": int(total_parameters),
    "trainable_parameters": int(trainable_parameters)
}

architecture_path = (
    Path(OUTPUT_DIRS["artifacts"])
    / "vgg16_architecture.json"
)

architecture_temp_path = (
    architecture_path.with_suffix(".json.tmp")
)

with open(
    architecture_temp_path,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        architecture_info,
        file,
        ensure_ascii=False,
        indent=2
    )

os.replace(
    architecture_temp_path,
    architecture_path
)


# ------------------------------------------------------------
# 9. Smoke modelini temizle ve gerçek modeli sıfırdan oluştur
# ------------------------------------------------------------

del smoke_model
del verification_model
del smoke_optimizer
del smoke_scheduler
del loaded_smoke_checkpoint
del smoke_images
del smoke_targets
del smoke_logits
del verification_logits

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Smoke test güncellemelerini gerçek modele taşımıyoruz.
set_all_seeds(SEED)

model = VGG16MouthClassifier(
    hidden_dim=HIDDEN_DIM,
    dropout=DROPOUT,
    number_of_classes=2
).to(DEVICE)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2
)

print("\nMODEL SUMMARY")
print("-" * 54)
print("Model: VGG16")
print("Başlangıç ağırlıkları: None")
print("Eğitim biçimi: Sıfırdan")
print("CNN özellik boyutu: 512")
print("Toplam parametre:", f"{total_parameters:,}")
print("Eğitilebilir parametre:", f"{trainable_parameters:,}")
print("Mimari kaydı:", architecture_path)

print("\n✅ Forward ve backward smoke testi başarılı.")
print("✅ Checkpoint atomik olarak kaydedildi.")
print("✅ Checkpoint yeniden yükleme testi başarılı.")
print("✅ Gerçek eğitim modeli seed=42 ile yeniden oluşturuldu.")
print("✅ Henüz gerçek eğitim başlatılmadı.")

DEVICE CHECK
------------------------------------------------------
Kullanılan cihaz: cuda
GPU: Tesla T4

SMOKE TEST
------------------------------------------------------
Girdi şekli: (4, 3, 128, 128)
Çıktı şekli: (4, 2)
Smoke loss: 0.670389

CHECKPOINT TEST
------------------------------------------------------
Checkpoint: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Dilara/Deney 1/Sonuçlar/VGG16_HOG_GIST_RBF_SVM_Mouth/20260807_1410_mouth_vgg16scratch_hog_gist_rbf_svm_seed42/checkpoints/smoke_test_checkpoint.pt
Eksik anahtar: 0
Yeniden yükleme: Başarılı

MODEL SUMMARY
------------------------------------------------------
Model: VGG16
Başlangıç ağırlıkları: None
Eğitim biçimi: Sıfırdan
CNN özellik boyutu: 512
Toplam parametre: 14,846,530
Eğitilebilir parametre: 14,846,530
Mimari kaydı: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Dilara/Deney 1/Sonuçlar/VGG16_HOG_GIST_RBF_SVM_Mouth/20260807_1410_mouth_vgg16scratch_hog_gist_rbf_svm_seed42/artifacts/vgg1

In [23]:
# ============================================================
# HÜCRE 15 — VGG16 GERÇEK EĞİTİMİ
# BEST/LAST CHECKPOINT + EARLY STOPPING + RESUME
# ============================================================

import os
import gc
import time
import random
import numpy as np
import pandas as pd
import torch

from pathlib import Path
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, f1_score


# ------------------------------------------------------------
# 1. Eğitim ayarları
# ------------------------------------------------------------

MAX_EPOCHS = int(getattr(CONFIG, "epochs", 30))
EARLY_STOPPING_PATIENCE = int(
    getattr(CONFIG, "early_stopping_patience", 7)
)
KEEP_LAST_CHECKPOINTS = int(
    getattr(CONFIG, "keep_last_checkpoints", 3)
)

USE_AMP = DEVICE.type == "cuda"

CHECKPOINT_DIR = Path(OUTPUT_DIRS["checkpoints"])
LOG_DIR = Path(OUTPUT_DIRS["logs"])
METRICS_DIR = Path(OUTPUT_DIRS["metrics"])

LAST_CHECKPOINT_PATH = (
    CHECKPOINT_DIR / "vgg16_last.pt"
)

BEST_CHECKPOINT_PATH = (
    CHECKPOINT_DIR / "vgg16_best.pt"
)

HISTORY_PATH = (
    LOG_DIR / "vgg16_training_history.csv"
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=USE_AMP
)


# ------------------------------------------------------------
# 2. Atomik PyTorch checkpoint kaydetme
# ------------------------------------------------------------

def atomic_torch_save(payload, destination):
    destination = Path(destination)
    temporary = destination.with_suffix(
        destination.suffix + ".tmp"
    )

    torch.save(payload, temporary)
    os.replace(temporary, destination)


# ------------------------------------------------------------
# 3. RNG durumları
# ------------------------------------------------------------

def capture_rng_state():
    state = {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch_cpu": torch.get_rng_state(),
        "loader_generator": loader_generator.get_state()
    }

    if torch.cuda.is_available():
        state["torch_cuda"] = (
            torch.cuda.get_rng_state_all()
        )

    return state


def restore_rng_state(state):
    if not state:
        return

    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch_cpu"])

    if "loader_generator" in state:
        loader_generator.set_state(
            state["loader_generator"]
        )

    if (
        torch.cuda.is_available()
        and "torch_cuda" in state
    ):
        torch.cuda.set_rng_state_all(
            state["torch_cuda"]
        )


# ------------------------------------------------------------
# 4. Tek epoch çalıştırma fonksiyonu
# ------------------------------------------------------------

def run_epoch(
    model,
    data_loader,
    criterion,
    device,
    optimizer=None,
    scaler=None,
    training=False
):
    if training:
        model.train()
        description = "Training"
    else:
        model.eval()
        description = "Validation"

    running_loss = 0.0
    all_targets = []
    all_predictions = []

    progress_bar = tqdm(
        data_loader,
        desc=description,
        leave=False
    )

    for batch in progress_bar:
        images = batch["image"].to(
            device,
            non_blocking=True
        )

        targets = batch["target"].to(
            device,
            non_blocking=True
        )

        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):
            with torch.amp.autocast(
                device_type=device.type,
                enabled=USE_AMP
            ):
                logits = model(images)
                loss = criterion(logits, targets)

            if training:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

        predictions = torch.argmax(
            logits,
            dim=1
        )

        batch_size = images.size(0)

        running_loss += (
            float(loss.item()) * batch_size
        )

        all_targets.extend(
            targets.detach().cpu().tolist()
        )

        all_predictions.extend(
            predictions.detach().cpu().tolist()
        )

        progress_bar.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    epoch_loss = (
        running_loss / len(data_loader.dataset)
    )

    epoch_accuracy = accuracy_score(
        all_targets,
        all_predictions
    )

    epoch_f1 = f1_score(
        all_targets,
        all_predictions,
        average="binary",
        pos_label=1,
        zero_division=0
    )

    return {
        "loss": float(epoch_loss),
        "accuracy": float(epoch_accuracy),
        "f1": float(epoch_f1)
    }


# ------------------------------------------------------------
# 5. Eğitim geçmişini atomik kaydet
# ------------------------------------------------------------

def save_training_history(history):
    history_frame = pd.DataFrame(history)

    temporary_path = HISTORY_PATH.with_suffix(
        ".csv.tmp"
    )

    history_frame.to_csv(
        temporary_path,
        index=False
    )

    os.replace(
        temporary_path,
        HISTORY_PATH
    )


# ------------------------------------------------------------
# 6. Eski epoch checkpointlerini son 3 ile sınırla
# ------------------------------------------------------------

def prune_epoch_checkpoints():
    epoch_checkpoints = sorted(
        CHECKPOINT_DIR.glob(
            "vgg16_epoch_*.pt"
        ),
        key=lambda path: path.stat().st_mtime
    )

    while (
        len(epoch_checkpoints)
        > KEEP_LAST_CHECKPOINTS
    ):
        oldest_checkpoint = epoch_checkpoints.pop(0)
        oldest_checkpoint.unlink()


# ------------------------------------------------------------
# 7. Başlangıç veya resume durumu
# ------------------------------------------------------------

start_epoch = 1
best_validation_f1 = -1.0
best_epoch = 0
epochs_without_improvement = 0
training_history = []

if LAST_CHECKPOINT_PATH.exists():
    print("Mevcut last checkpoint bulundu.")
    print("Eğitime kaldığı yerden devam ediliyor.")

    resume_checkpoint = torch.load(
        LAST_CHECKPOINT_PATH,
        map_location=DEVICE,
        weights_only=False
    )

    if resume_checkpoint["run_id"] != str(RUN_ID):
        raise RuntimeError(
            "Checkpoint run_id mevcut deneyle uyuşmuyor."
        )

    model.load_state_dict(
        resume_checkpoint["model_state_dict"]
    )

    optimizer.load_state_dict(
        resume_checkpoint["optimizer_state_dict"]
    )

    scheduler.load_state_dict(
        resume_checkpoint["scheduler_state_dict"]
    )

    scaler.load_state_dict(
        resume_checkpoint["scaler_state_dict"]
    )

    restore_rng_state(
        resume_checkpoint.get("rng_state")
    )

    start_epoch = (
        int(resume_checkpoint["epoch"]) + 1
    )

    best_validation_f1 = float(
        resume_checkpoint["best_validation_f1"]
    )

    best_epoch = int(
        resume_checkpoint["best_epoch"]
    )

    epochs_without_improvement = int(
        resume_checkpoint[
            "epochs_without_improvement"
        ]
    )

    training_history = resume_checkpoint.get(
        "training_history",
        []
    )

    del resume_checkpoint
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

else:
    print("Yeni VGG16 eğitimi başlatılıyor.")
    print("Önceden eğitilmiş ağırlık kullanılmıyor.")


print("\nTRAINING CONFIGURATION")
print("-" * 58)
print("Başlangıç epoch:", start_epoch)
print("Maksimum epoch:", MAX_EPOCHS)
print("Early stopping patience:", EARLY_STOPPING_PATIENCE)
print("Seçim metriği: Validation F1")
print("AMP aktif:", USE_AMP)
print("Test spliti eğitimde kullanılmayacak.")
print("Checkpoint klasörü:", CHECKPOINT_DIR)


# ------------------------------------------------------------
# 8. Gerçek eğitim döngüsü
# ------------------------------------------------------------

training_start_time = time.time()

for epoch in range(start_epoch, MAX_EPOCHS + 1):
    epoch_start_time = time.time()

    print(
        f"\nEpoch {epoch:02d}/{MAX_EPOCHS}"
    )
    print("-" * 58)

    train_metrics = run_epoch(
        model=model,
        data_loader=train_loader,
        criterion=criterion,
        device=DEVICE,
        optimizer=optimizer,
        scaler=scaler,
        training=True
    )

    validation_metrics = run_epoch(
        model=model,
        data_loader=val_loader,
        criterion=criterion,
        device=DEVICE,
        optimizer=None,
        scaler=None,
        training=False
    )

    scheduler.step(
        validation_metrics["loss"]
    )

    current_learning_rate = float(
        optimizer.param_groups[0]["lr"]
    )

    epoch_duration_seconds = (
        time.time() - epoch_start_time
    )

    history_row = {
        "epoch": int(epoch),
        "train_loss": train_metrics["loss"],
        "train_accuracy": train_metrics["accuracy"],
        "train_f1": train_metrics["f1"],
        "validation_loss": validation_metrics["loss"],
        "validation_accuracy": validation_metrics["accuracy"],
        "validation_f1": validation_metrics["f1"],
        "learning_rate": current_learning_rate,
        "epoch_seconds": float(
            epoch_duration_seconds
        )
    }

    training_history.append(history_row)
    save_training_history(training_history)

    improved = (
        validation_metrics["f1"]
        > best_validation_f1
    )

    if improved:
        best_validation_f1 = (
            validation_metrics["f1"]
        )
        best_epoch = epoch
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    checkpoint_payload = {
        "checkpoint_type": "training",
        "run_id": str(RUN_ID),
        "epoch": int(epoch),
        "seed": SEED,

        "model_name": "VGG16_from_scratch",
        "weights": None,

        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(),

        "best_validation_f1": float(
            best_validation_f1
        ),
        "best_epoch": int(best_epoch),

        "epochs_without_improvement": int(
            epochs_without_improvement
        ),

        "training_history": training_history,
        "rng_state": capture_rng_state(),

        "class_mapping": {
            "real": 0,
            "fake": 1
        }
    }

    # Her epoch sonundaki güncel durum
    atomic_torch_save(
        checkpoint_payload,
        LAST_CHECKPOINT_PATH
    )

    # Son üç epoch için dönen checkpoint
    epoch_checkpoint_path = (
        CHECKPOINT_DIR
        / f"vgg16_epoch_{epoch:03d}.pt"
    )

    atomic_torch_save(
        checkpoint_payload,
        epoch_checkpoint_path
    )

    prune_epoch_checkpoints()

    # En iyi validation F1 checkpointi
    if improved:
        best_payload = checkpoint_payload.copy()
        best_payload["checkpoint_type"] = "best"

        atomic_torch_save(
            best_payload,
            BEST_CHECKPOINT_PATH
        )

    print(
        f"Train Loss: {train_metrics['loss']:.4f} | "
        f"Train Acc: {train_metrics['accuracy']:.4f} | "
        f"Train F1: {train_metrics['f1']:.4f}"
    )

    print(
        f"Val Loss:   {validation_metrics['loss']:.4f} | "
        f"Val Acc:   {validation_metrics['accuracy']:.4f} | "
        f"Val F1:   {validation_metrics['f1']:.4f}"
    )

    print(
        f"LR: {current_learning_rate:.8f} | "
        f"Süre: {epoch_duration_seconds / 60:.2f} dk"
    )

    if improved:
        print(
            "✅ Yeni en iyi model kaydedildi. "
            f"Validation F1: {best_validation_f1:.4f}"
        )
    else:
        print(
            "İyileşmeyen epoch: "
            f"{epochs_without_improvement}/"
            f"{EARLY_STOPPING_PATIENCE}"
        )

    if (
        epochs_without_improvement
        >= EARLY_STOPPING_PATIENCE
    ):
        print("\nEarly stopping etkinleşti.")
        print(
            f"En iyi epoch: {best_epoch} | "
            f"En iyi Validation F1: "
            f"{best_validation_f1:.4f}"
        )
        break


# ------------------------------------------------------------
# 9. Eğitim sonu özeti
# ------------------------------------------------------------

total_training_seconds = (
    time.time() - training_start_time
)

assert BEST_CHECKPOINT_PATH.exists()
assert LAST_CHECKPOINT_PATH.exists()
assert HISTORY_PATH.exists()

print("\nVGG16 TRAINING COMPLETED")
print("=" * 58)
print("En iyi epoch:", best_epoch)
print(
    "En iyi Validation F1:",
    round(best_validation_f1, 6)
)
print(
    "Bu oturumdaki süre:",
    f"{total_training_seconds / 60:.2f} dakika"
)
print("Best checkpoint:", BEST_CHECKPOINT_PATH)
print("Last checkpoint:", LAST_CHECKPOINT_PATH)
print("Eğitim geçmişi:", HISTORY_PATH)

print("\n✅ Eğitim sırasında yalnızca train kullanıldı.")
print("✅ Model seçimi yalnızca validation F1 ile yapıldı.")
print("✅ Test splitine henüz dokunulmadı.")
print("✅ Best/last ve son 3 epoch checkpointi kaydedildi.")

Yeni VGG16 eğitimi başlatılıyor.
Önceden eğitilmiş ağırlık kullanılmıyor.

TRAINING CONFIGURATION
----------------------------------------------------------
Başlangıç epoch: 1
Maksimum epoch: 30
Early stopping patience: 7
Seçim metriği: Validation F1
AMP aktif: True
Test spliti eğitimde kullanılmayacak.
Checkpoint klasörü: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Dilara/Deney 1/Sonuçlar/VGG16_HOG_GIST_RBF_SVM_Mouth/20260807_1410_mouth_vgg16scratch_hog_gist_rbf_svm_seed42/checkpoints

Epoch 01/30
----------------------------------------------------------


Training:   0%|          | 0/73 [00:00<?, ?it/s]

Validation:   0%|          | 0/9 [00:00<?, ?it/s]

Train Loss: 0.6939 | Train Acc: 0.5026 | Train F1: 0.5915
Val Loss:   0.6936 | Val Acc:   0.4878 | Val F1:   0.6240
LR: 0.00010000 | Süre: 0.40 dk
✅ Yeni en iyi model kaydedildi. Validation F1: 0.6240

Epoch 02/30
----------------------------------------------------------


Training:   0%|          | 0/73 [00:00<?, ?it/s]

...

Validation:   0%|          | 0/9 [00:00<?, ?it/s]

Train Loss: 0.6661 | Train Acc: 0.6004 | Train F1: 0.6156
Val Loss:   0.6701 | Val Acc:   0.6167 | Val F1:   0.6014
LR: 0.00010000 | Süre: 0.43 dk
İyileşmeyen epoch: 6/7

Epoch 10/30
----------------------------------------------------------


Training:   0%|          | 0/73 [00:00<?, ?it/s]

Validation:   0%|          | 0/9 [00:00<?, ?it/s]

Train Loss: 0.6566 | Train Acc: 0.6177 | Train F1: 0.6156
Val Loss:   0.6728 | Val Acc:   0.5436 | Val F1:   0.3282
LR: 0.00010000 | Süre: 0.42 dk
İyileşmeyen epoch: 7/7

Early stopping etkinleşti.
En iyi epoch: 3 | En iyi Validation F1: 0.6462

VGG16 TRAINING COMPLETED
En iyi epoch: 3
En iyi Validation F1: 0.646226
Bu oturumdaki süre: 4.50 dakika
Best checkpoint: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Dilara/Deney 1/Sonuçlar/VGG16_HOG_GIST_RBF_SVM_Mouth/20260807_1410_mouth_vgg16scratch_hog_gist_rbf_svm_seed42/checkpoints/vgg16_best.pt
Last checkpoint: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Dilara/Deney 1/Sonuçlar/VGG16_HOG_GIST_RBF_SVM_Mouth/20260807_1410_mouth_vgg16scratch_hog_gist_rbf_svm_seed42/checkpoints/vgg16_last.pt
Eğitim geçmişi: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Dilara/Deney 1/Sonuçlar/VGG16_HOG_GIST_RBF_SVM_Mouth/20260807_1410_mouth_vgg16scratch_hog_gist_rbf_svm_seed42/logs/vgg16_training_history.csv

✅ 

In [24]:
# ============================================================
# HÜCRE 16 — EĞİTİM GRAFİKLERİ VE BEST MODELİ YÜKLEME
# ============================================================

import os
import json
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from pathlib import Path
from PIL import Image


# ------------------------------------------------------------
# 1. Dosya yolları
# ------------------------------------------------------------

HISTORY_PATH = (
    Path(OUTPUT_DIRS["logs"])
    / "vgg16_training_history.csv"
)

BEST_CHECKPOINT_PATH = (
    Path(OUTPUT_DIRS["checkpoints"])
    / "vgg16_best.pt"
)

TRAINING_FIGURE_PATH = (
    Path(OUTPUT_DIRS["figures"])
    / "vgg16_training_curves.png"
)

TRAINING_SUMMARY_PATH = (
    Path(OUTPUT_DIRS["metrics"])
    / "vgg16_training_summary.json"
)

assert HISTORY_PATH.exists(), (
    f"Eğitim geçmişi bulunamadı: {HISTORY_PATH}"
)

assert BEST_CHECKPOINT_PATH.exists(), (
    f"Best checkpoint bulunamadı: "
    f"{BEST_CHECKPOINT_PATH}"
)


# ------------------------------------------------------------
# 2. Eğitim geçmişini yükle
# ------------------------------------------------------------

history_df = pd.read_csv(HISTORY_PATH)

required_columns = {
    "epoch",
    "train_loss",
    "train_accuracy",
    "train_f1",
    "validation_loss",
    "validation_accuracy",
    "validation_f1",
    "learning_rate",
    "epoch_seconds"
}

missing_columns = (
    required_columns
    - set(history_df.columns)
)

if missing_columns:
    raise RuntimeError(
        f"Eğitim geçmişinde eksik sütunlar: "
        f"{missing_columns}"
    )

best_history_index = (
    history_df["validation_f1"].idxmax()
)

best_history_row = history_df.loc[
    best_history_index
]

best_epoch_from_history = int(
    best_history_row["epoch"]
)

best_validation_f1_from_history = float(
    best_history_row["validation_f1"]
)


# ------------------------------------------------------------
# 3. Eğitim grafikleri
# ------------------------------------------------------------

plt.style.use("seaborn-v0_8-whitegrid")

figure, axes = plt.subplots(
    2,
    2,
    figsize=(12, 8)
)

epochs = history_df["epoch"].to_numpy()


# Loss
axes[0, 0].plot(
    epochs,
    history_df["train_loss"],
    marker="o",
    linewidth=2,
    label="Train Loss",
    color="#1f77b4"
)

axes[0, 0].plot(
    epochs,
    history_df["validation_loss"],
    marker="o",
    linewidth=2,
    label="Validation Loss",
    color="#ff7f0e"
)

axes[0, 0].axvline(
    best_epoch_from_history,
    linestyle="--",
    linewidth=1.5,
    color="#2ca02c",
    label=f"Best Epoch ({best_epoch_from_history})"
)

axes[0, 0].set_title("VGG16 Loss")
axes[0, 0].set_xlabel("Epoch")
axes[0, 0].set_ylabel("Cross-Entropy Loss")
axes[0, 0].legend()


# Accuracy
axes[0, 1].plot(
    epochs,
    history_df["train_accuracy"],
    marker="o",
    linewidth=2,
    label="Train Accuracy",
    color="#1f77b4"
)

axes[0, 1].plot(
    epochs,
    history_df["validation_accuracy"],
    marker="o",
    linewidth=2,
    label="Validation Accuracy",
    color="#ff7f0e"
)

axes[0, 1].axvline(
    best_epoch_from_history,
    linestyle="--",
    linewidth=1.5,
    color="#2ca02c"
)

axes[0, 1].set_title("VGG16 Accuracy")
axes[0, 1].set_xlabel("Epoch")
axes[0, 1].set_ylabel("Accuracy")
axes[0, 1].set_ylim(0.0, 1.0)
axes[0, 1].legend()


# F1
axes[1, 0].plot(
    epochs,
    history_df["train_f1"],
    marker="o",
    linewidth=2,
    label="Train F1",
    color="#1f77b4"
)

axes[1, 0].plot(
    epochs,
    history_df["validation_f1"],
    marker="o",
    linewidth=2,
    label="Validation F1",
    color="#ff7f0e"
)

axes[1, 0].scatter(
    [best_epoch_from_history],
    [best_validation_f1_from_history],
    s=100,
    color="#2ca02c",
    zorder=5,
    label=(
        f"Best Validation F1 "
        f"({best_validation_f1_from_history:.4f})"
    )
)

axes[1, 0].set_title("VGG16 F1 Score")
axes[1, 0].set_xlabel("Epoch")
axes[1, 0].set_ylabel("F1 Score")
axes[1, 0].set_ylim(0.0, 1.0)
axes[1, 0].legend()


# Learning rate
axes[1, 1].plot(
    epochs,
    history_df["learning_rate"],
    marker="o",
    linewidth=2,
    color="#9467bd"
)

axes[1, 1].set_title("Learning Rate Schedule")
axes[1, 1].set_xlabel("Epoch")
axes[1, 1].set_ylabel("Learning Rate")
axes[1, 1].ticklabel_format(
    axis="y",
    style="scientific",
    scilimits=(0, 0)
)


figure.suptitle(
    "VGG16 From-Scratch Training on Mouth ROI",
    fontsize=15,
    fontweight="bold"
)

figure.tight_layout(
    rect=[0, 0, 1, 0.96]
)

temporary_figure_path = (
    TRAINING_FIGURE_PATH.with_suffix(
        ".png.tmp"
    )
)

figure.savefig(
    temporary_figure_path,
    format="png",
    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)

plt.close(figure)

os.replace(
    temporary_figure_path,
    TRAINING_FIGURE_PATH
)


# ------------------------------------------------------------
# 4. Grafik kalite kontrolü
# ------------------------------------------------------------

with Image.open(TRAINING_FIGURE_PATH) as image:
    figure_width, figure_height = image.size

assert min(
    figure_width,
    figure_height
) >= 600


# ------------------------------------------------------------
# 5. En iyi checkpoint'i yükle
# ------------------------------------------------------------

best_checkpoint = torch.load(
    BEST_CHECKPOINT_PATH,
    map_location=DEVICE,
    weights_only=False
)

if best_checkpoint["run_id"] != str(RUN_ID):
    raise RuntimeError(
        "Best checkpoint farklı bir run'a ait."
    )

model.load_state_dict(
    best_checkpoint["model_state_dict"]
)

model = model.to(DEVICE)
model.eval()

best_epoch = int(
    best_checkpoint["best_epoch"]
)

best_validation_f1 = float(
    best_checkpoint["best_validation_f1"]
)

assert best_epoch == best_epoch_from_history

assert np.isclose(
    best_validation_f1,
    best_validation_f1_from_history
)


# ------------------------------------------------------------
# 6. Yüklenen modeli validation batch ile doğrula
# ------------------------------------------------------------

validation_images = (
    first_val_batch["image"][:4]
    .to(DEVICE)
)

with torch.no_grad():
    validation_logits = model(
        validation_images
    )

    validation_probabilities = torch.softmax(
        validation_logits,
        dim=1
    )

assert validation_logits.shape == (4, 2)
assert torch.isfinite(
    validation_probabilities
).all()


# ------------------------------------------------------------
# 7. Eğitim özeti
# ------------------------------------------------------------

training_summary = {
    "run_id": str(RUN_ID),
    "model": "VGG16_from_scratch",
    "roi": "mouth",
    "epochs_completed": int(len(history_df)),
    "maximum_epochs": int(MAX_EPOCHS),
    "early_stopping_patience": int(
        EARLY_STOPPING_PATIENCE
    ),
    "early_stopping_triggered": bool(
        len(history_df) < MAX_EPOCHS
    ),
    "selection_metric": "validation_f1",
    "best_epoch": best_epoch,
    "best_validation_f1": (
        best_validation_f1
    ),
    "best_validation_accuracy": float(
        history_df.loc[
            history_df["epoch"].eq(best_epoch),
            "validation_accuracy"
        ].iloc[0]
    ),
    "best_validation_loss": float(
        history_df.loc[
            history_df["epoch"].eq(best_epoch),
            "validation_loss"
        ].iloc[0]
    ),
    "test_used_during_training": False,
    "figure_dpi": 600,
    "figure_size_pixels": [
        int(figure_width),
        int(figure_height)
    ],
    "best_checkpoint": str(
        BEST_CHECKPOINT_PATH
    )
}

temporary_summary_path = (
    TRAINING_SUMMARY_PATH.with_suffix(
        ".json.tmp"
    )
)

with open(
    temporary_summary_path,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        training_summary,
        file,
        ensure_ascii=False,
        indent=2
    )

os.replace(
    temporary_summary_path,
    TRAINING_SUMMARY_PATH
)


# ------------------------------------------------------------
# 8. Sonuç
# ------------------------------------------------------------

print("VGG16 TRAINING ARTIFACTS")
print("-" * 58)

print("Tamamlanan epoch:", len(history_df))
print("Yüklenen best epoch:", best_epoch)

print(
    "Best Validation F1:",
    round(best_validation_f1, 6)
)

print(
    "Grafik çözünürlüğü:",
    f"{figure_width}x{figure_height}"
)

print("Grafik DPI: 600")
print("Eğitim grafiği:", TRAINING_FIGURE_PATH)
print("Eğitim özeti:", TRAINING_SUMMARY_PATH)
print("Best checkpoint:", BEST_CHECKPOINT_PATH)

print("\n✅ En iyi VGG16 checkpoint'i yüklendi.")
print("✅ Eğitim grafikleri tamamen İngilizce.")
print("✅ Grafik 600 DPI olarak kaydedildi.")
print("✅ Test verisi hâlâ kullanılmadı.")

VGG16 TRAINING ARTIFACTS
----------------------------------------------------------
Tamamlanan epoch: 10
Yüklenen best epoch: 3
Best Validation F1: 0.646226
Grafik çözünürlüğü: 7133x4722
Grafik DPI: 600
Eğitim grafiği: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Dilara/Deney 1/Sonuçlar/VGG16_HOG_GIST_RBF_SVM_Mouth/20260807_1410_mouth_vgg16scratch_hog_gist_rbf_svm_seed42/figures/vgg16_training_curves.png
Eğitim özeti: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Dilara/Deney 1/Sonuçlar/VGG16_HOG_GIST_RBF_SVM_Mouth/20260807_1410_mouth_vgg16scratch_hog_gist_rbf_svm_seed42/metrics/vgg16_training_summary.json
Best checkpoint: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Dilara/Deney 1/Sonuçlar/VGG16_HOG_GIST_RBF_SVM_Mouth/20260807_1410_mouth_vgg16scratch_hog_gist_rbf_svm_seed42/checkpoints/vgg16_best.pt

✅ En iyi VGG16 checkpoint'i yüklendi.
✅ Eğitim grafikleri tamamen İngilizce.
✅ Grafik 600 DPI olarak kaydedildi.
✅ Test verisi hâlâ kullanıl

In [25]:
# ============================================================
# HÜCRE 17 — VGG16 + HOG + GIST ÖZELLİK ÇIKARIMI
# ============================================================

import os
import time
import json
import cv2
import numpy as np
import torch

from pathlib import Path
from tqdm.auto import tqdm
from skimage.feature import hog


# ------------------------------------------------------------
# 1. Özellik ayarları
# ------------------------------------------------------------

FEATURE_DIR = (
    Path(OUTPUT_DIRS["artifacts"])
    / "feature_cache"
)

FEATURE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CNN_FEATURE_DIM = 512

# 64x128 görüntü + 8x8 hücre + 2x2 blok + 9 yön
# sonucunda 3780 HOG özelliği elde edilir.
HOG_WIDTH = 64
HOG_HEIGHT = 128
HOG_ORIENTATIONS = 9
HOG_PIXELS_PER_CELL = (8, 8)
HOG_CELLS_PER_BLOCK = (2, 2)
HOG_FEATURE_DIM = 3780

GIST_IMAGE_SIZE = 128
GIST_GRID_SIZE = 4
GIST_ORIENTATIONS = 8

# (sigma, wavelength)
GIST_SCALE_PARAMETERS = [
    (2.0, 4.0),
    (3.0, 6.0),
    (4.0, 8.0),
    (5.0, 10.0)
]

GIST_FEATURE_DIM = (
    len(GIST_SCALE_PARAMETERS)
    * GIST_ORIENTATIONS
    * GIST_GRID_SIZE
    * GIST_GRID_SIZE
)

assert GIST_FEATURE_DIM == 512

model.eval()


# ------------------------------------------------------------
# 2. Gabor filtre bankası
# ------------------------------------------------------------

def build_gabor_filter_bank():
    filters = []

    for sigma, wavelength in GIST_SCALE_PARAMETERS:
        for orientation_index in range(
            GIST_ORIENTATIONS
        ):
            theta = (
                orientation_index
                * np.pi
                / GIST_ORIENTATIONS
            )

            kernel = cv2.getGaborKernel(
                ksize=(31, 31),
                sigma=sigma,
                theta=theta,
                lambd=wavelength,
                gamma=0.5,
                psi=0,
                ktype=cv2.CV_32F
            )

            kernel_norm = np.sum(
                np.abs(kernel)
            )

            if kernel_norm > 0:
                kernel = kernel / kernel_norm

            filters.append(
                kernel.astype(np.float32)
            )

    return filters


GABOR_FILTER_BANK = build_gabor_filter_bank()

assert len(GABOR_FILTER_BANK) == 32


# ------------------------------------------------------------
# 3. HOG özellik çıkarımı
# ------------------------------------------------------------

def extract_hog_features(rgb_image):
    gray_image = cv2.cvtColor(
        rgb_image,
        cv2.COLOR_RGB2GRAY
    )

    hog_image = cv2.resize(
        gray_image,
        (HOG_WIDTH, HOG_HEIGHT),
        interpolation=cv2.INTER_AREA
    )

    feature_vector = hog(
        hog_image,
        orientations=HOG_ORIENTATIONS,
        pixels_per_cell=HOG_PIXELS_PER_CELL,
        cells_per_block=HOG_CELLS_PER_BLOCK,
        block_norm="L2-Hys",
        transform_sqrt=True,
        feature_vector=True
    )

    feature_vector = np.asarray(
        feature_vector,
        dtype=np.float32
    )

    if feature_vector.shape[0] != HOG_FEATURE_DIM:
        raise RuntimeError(
            "Beklenmeyen HOG boyutu: "
            f"{feature_vector.shape[0]}"
        )

    return feature_vector


# ------------------------------------------------------------
# 4. GIST/Gabor özellik çıkarımı
# ------------------------------------------------------------

def extract_gist_features(rgb_image):
    gray_image = cv2.cvtColor(
        rgb_image,
        cv2.COLOR_RGB2GRAY
    )

    gray_image = cv2.resize(
        gray_image,
        (GIST_IMAGE_SIZE, GIST_IMAGE_SIZE),
        interpolation=cv2.INTER_AREA
    )

    gray_image = (
        gray_image.astype(np.float32) / 255.0
    )

    cell_height = (
        GIST_IMAGE_SIZE // GIST_GRID_SIZE
    )

    cell_width = (
        GIST_IMAGE_SIZE // GIST_GRID_SIZE
    )

    pooled_features = []

    for gabor_kernel in GABOR_FILTER_BANK:
        response = cv2.filter2D(
            gray_image,
            cv2.CV_32F,
            gabor_kernel,
            borderType=cv2.BORDER_REFLECT
        )

        response = np.abs(response)

        for grid_y in range(GIST_GRID_SIZE):
            for grid_x in range(GIST_GRID_SIZE):
                y_start = grid_y * cell_height
                y_end = (
                    (grid_y + 1) * cell_height
                )

                x_start = grid_x * cell_width
                x_end = (
                    (grid_x + 1) * cell_width
                )

                grid_cell = response[
                    y_start:y_end,
                    x_start:x_end
                ]

                pooled_features.append(
                    float(np.mean(grid_cell))
                )

    feature_vector = np.asarray(
        pooled_features,
        dtype=np.float32
    )

    if feature_vector.shape[0] != GIST_FEATURE_DIM:
        raise RuntimeError(
            "Beklenmeyen GIST boyutu: "
            f"{feature_vector.shape[0]}"
        )

    return feature_vector


# ------------------------------------------------------------
# 5. NPZ dosyasını atomik kaydet
# ------------------------------------------------------------

def atomic_save_npz(destination, **arrays):
    destination = Path(destination)

    temporary_path = destination.with_suffix(
        ".npz.tmp"
    )

    with open(temporary_path, "wb") as file:
        np.savez_compressed(
            file,
            **arrays
        )

    os.replace(
        temporary_path,
        destination
    )


# ------------------------------------------------------------
# 6. Bir split için özellik çıkarımı
# ------------------------------------------------------------

def extract_split_features(
    split_name,
    split_frame,
    cnn_batch_size=32
):
    cache_path = (
        FEATURE_DIR
        / f"{split_name}_features.npz"
    )

    # Hücre tekrar çalıştırılırsa tamamlanmış cache kullanılır.
    if cache_path.exists():
        print(
            f"\n{split_name.upper()} cache bulundu, "
            "yeniden hesaplanmıyor."
        )

        cached = np.load(
            cache_path,
            allow_pickle=False
        )

        required_keys = {
            "cnn",
            "hog",
            "gist",
            "labels",
            "file_names",
            "source_videos",
            "image_paths"
        }

        if not required_keys.issubset(
            set(cached.files)
        ):
            cached.close()
            raise RuntimeError(
                f"{split_name} cache eksik veya bozuk."
            )

        result = {
            key: cached[key]
            for key in required_keys
        }

        cached.close()

        expected_count = len(split_frame)

        assert result["cnn"].shape == (
            expected_count,
            CNN_FEATURE_DIM
        )

        assert result["hog"].shape == (
            expected_count,
            HOG_FEATURE_DIM
        )

        assert result["gist"].shape == (
            expected_count,
            GIST_FEATURE_DIM
        )

        return result

    print(
        f"\n{split_name.upper()} özellikleri "
        "çıkarılıyor..."
    )

    split_start_time = time.time()

    cnn_features = []
    hog_features = []
    gist_features = []

    labels = []
    file_names = []
    source_videos = []
    image_paths = []

    pending_cnn_tensors = []

    def process_cnn_batch():
        if len(pending_cnn_tensors) == 0:
            return

        batch_tensor = torch.stack(
            pending_cnn_tensors,
            dim=0
        ).to(
            DEVICE,
            non_blocking=True
        )

        with torch.no_grad():
            with torch.amp.autocast(
                device_type=DEVICE.type,
                enabled=(DEVICE.type == "cuda")
            ):
                batch_features = (
                    model.forward_features(
                        batch_tensor
                    )
                )

        batch_features = (
            batch_features
            .detach()
            .float()
            .cpu()
            .numpy()
            .astype(np.float32)
        )

        cnn_features.extend(
            list(batch_features)
        )

        pending_cnn_tensors.clear()

    progress_bar = tqdm(
        split_frame.itertuples(index=False),
        total=len(split_frame),
        desc=f"{split_name} feature extraction"
    )

    for row in progress_bar:
        image_path = str(
            getattr(row, "output_path")
        )

        rgb_image = letterbox_rgb(
            image_path=image_path,
            target_size=IMAGE_SIZE
        )

        hog_vector = extract_hog_features(
            rgb_image
        )

        gist_vector = extract_gist_features(
            rgb_image
        )

        pil_image = Image.fromarray(
            rgb_image
        )

        cnn_tensor = evaluation_transform(
            pil_image
        )

        pending_cnn_tensors.append(
            cnn_tensor
        )

        hog_features.append(hog_vector)
        gist_features.append(gist_vector)

        label_text = str(
            getattr(row, "label")
        ).strip().lower()

        if label_text == "real":
            numeric_label = 0
        elif label_text == "fake":
            numeric_label = 1
        else:
            raise ValueError(
                f"Bilinmeyen etiket: {label_text}"
            )

        labels.append(numeric_label)

        file_names.append(
            str(getattr(row, "file_name"))
        )

        source_videos.append(
            str(getattr(row, "source_video"))
        )

        image_paths.append(image_path)

        if (
            len(pending_cnn_tensors)
            >= cnn_batch_size
        ):
            process_cnn_batch()

    process_cnn_batch()

    result = {
        "cnn": np.asarray(
            cnn_features,
            dtype=np.float32
        ),
        "hog": np.asarray(
            hog_features,
            dtype=np.float32
        ),
        "gist": np.asarray(
            gist_features,
            dtype=np.float32
        ),
        "labels": np.asarray(
            labels,
            dtype=np.int64
        ),
        "file_names": np.asarray(
            file_names,
            dtype=str
        ),
        "source_videos": np.asarray(
            source_videos,
            dtype=str
        ),
        "image_paths": np.asarray(
            image_paths,
            dtype=str
        )
    }

    expected_count = len(split_frame)

    assert result["cnn"].shape == (
        expected_count,
        CNN_FEATURE_DIM
    )

    assert result["hog"].shape == (
        expected_count,
        HOG_FEATURE_DIM
    )

    assert result["gist"].shape == (
        expected_count,
        GIST_FEATURE_DIM
    )

    assert np.isfinite(result["cnn"]).all()
    assert np.isfinite(result["hog"]).all()
    assert np.isfinite(result["gist"]).all()

    atomic_save_npz(
        cache_path,
        **result
    )

    split_minutes = (
        time.time() - split_start_time
    ) / 60.0

    print(
        f"{split_name.upper()} tamamlandı: "
        f"{split_minutes:.2f} dakika"
    )

    print("Kaydedildi:", cache_path)

    return result


# ------------------------------------------------------------
# 7. Train, validation ve test özelliklerini çıkar
# ------------------------------------------------------------

feature_extraction_start = time.time()

train_features = extract_split_features(
    split_name="train",
    split_frame=train_frame
)

validation_features = extract_split_features(
    split_name="validation",
    split_frame=val_frame
)

test_features = extract_split_features(
    split_name="test",
    split_frame=test_frame
)


# ------------------------------------------------------------
# 8. Özellik boyutu ve veri sırası kontrolleri
# ------------------------------------------------------------

assert train_features["cnn"].shape == (
    2310,
    512
)

assert train_features["hog"].shape == (
    2310,
    3780
)

assert train_features["gist"].shape == (
    2310,
    512
)

assert validation_features["labels"].shape[0] == 287
assert test_features["labels"].shape[0] == 292

assert np.array_equal(
    train_features["labels"],
    train_frame["label"]
    .astype(str)
    .str.lower()
    .map({"real": 0, "fake": 1})
    .to_numpy(dtype=np.int64)
)

assert np.array_equal(
    validation_features["labels"],
    val_frame["label"]
    .astype(str)
    .str.lower()
    .map({"real": 0, "fake": 1})
    .to_numpy(dtype=np.int64)
)

assert np.array_equal(
    test_features["labels"],
    test_frame["label"]
    .astype(str)
    .str.lower()
    .map({"real": 0, "fake": 1})
    .to_numpy(dtype=np.int64)
)


# ------------------------------------------------------------
# 9. Özellik çıkarım ayarlarını kaydet
# ------------------------------------------------------------

feature_config = {
    "run_id": str(RUN_ID),
    "cnn": {
        "model": "best_validation_f1_vgg16",
        "weights_initialization": None,
        "feature_dimension": 512,
        "input_size": [128, 128],
        "augmentation_used": False
    },
    "hog": {
        "input_size": [
            HOG_HEIGHT,
            HOG_WIDTH
        ],
        "orientations": HOG_ORIENTATIONS,
        "pixels_per_cell": list(
            HOG_PIXELS_PER_CELL
        ),
        "cells_per_block": list(
            HOG_CELLS_PER_BLOCK
        ),
        "block_norm": "L2-Hys",
        "transform_sqrt": True,
        "feature_dimension": HOG_FEATURE_DIM
    },
    "gist": {
        "implementation": (
            "Gabor filter bank with grid pooling"
        ),
        "input_size": [
            GIST_IMAGE_SIZE,
            GIST_IMAGE_SIZE
        ],
        "scales_sigma_wavelength": [
            list(values)
            for values in GIST_SCALE_PARAMETERS
        ],
        "orientations": GIST_ORIENTATIONS,
        "grid_size": [
            GIST_GRID_SIZE,
            GIST_GRID_SIZE
        ],
        "feature_dimension": GIST_FEATURE_DIM
    },
    "raw_combined_dimension": (
        CNN_FEATURE_DIM
        + HOG_FEATURE_DIM
        + GIST_FEATURE_DIM
    ),
    "split_counts": {
        "train": len(train_frame),
        "validation": len(val_frame),
        "test": len(test_frame)
    },
    "test_labels_used_for_model_selection": False
}

feature_config_path = (
    Path(OUTPUT_DIRS["artifacts"])
    / "feature_extraction_config.json"
)

feature_config_temp = (
    feature_config_path.with_suffix(
        ".json.tmp"
    )
)

with open(
    feature_config_temp,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        feature_config,
        file,
        ensure_ascii=False,
        indent=2
    )

os.replace(
    feature_config_temp,
    feature_config_path
)


# ------------------------------------------------------------
# 10. Sonuç
# ------------------------------------------------------------

total_feature_minutes = (
    time.time() - feature_extraction_start
) / 60.0

print("\nFEATURE EXTRACTION COMPLETED")
print("=" * 60)

print(
    "Train CNN / HOG / GIST:",
    train_features["cnn"].shape,
    train_features["hog"].shape,
    train_features["gist"].shape
)

print(
    "Validation CNN / HOG / GIST:",
    validation_features["cnn"].shape,
    validation_features["hog"].shape,
    validation_features["gist"].shape
)

print(
    "Test CNN / HOG / GIST:",
    test_features["cnn"].shape,
    test_features["hog"].shape,
    test_features["gist"].shape
)

print(
    "Ham birleşik özellik boyutu:",
    CNN_FEATURE_DIM
    + HOG_FEATURE_DIM
    + GIST_FEATURE_DIM
)

print(
    "Toplam süre:",
    f"{total_feature_minutes:.2f} dakika"
)

print("Cache klasörü:", FEATURE_DIR)

print("\n✅ VGG16 özellikleri çıkarıldı.")
print("✅ HOG özellikleri çıkarıldı.")
print("✅ GIST/Gabor özellikleri çıkarıldı.")
print("✅ Özellikler split bazında Drive'a kaydedildi.")
print("✅ Model seçimi için test etiketi kullanılmadı.")


TRAIN özellikleri çıkarılıyor...


train feature extraction:   0%|          | 0/2310 [00:00<?, ?it/s]

TRAIN tamamlandı: 2.93 dakika
Kaydedildi: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Dilara/Deney 1/Sonuçlar/VGG16_HOG_GIST_RBF_SVM_Mouth/20260807_1410_mouth_vgg16scratch_hog_gist_rbf_svm_seed42/artifacts/feature_cache/train_features.npz

VALIDATION özellikleri çıkarılıyor...


validation feature extraction:   0%|          | 0/287 [00:00<?, ?it/s]

VALIDATION tamamlandı: 0.35 dakika
Kaydedildi: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Dilara/Deney 1/Sonuçlar/VGG16_HOG_GIST_RBF_SVM_Mouth/20260807_1410_mouth_vgg16scratch_hog_gist_rbf_svm_seed42/artifacts/feature_cache/validation_features.npz

TEST özellikleri çıkarılıyor...


test feature extraction:   0%|          | 0/292 [00:00<?, ?it/s]

TEST tamamlandı: 0.34 dakika
Kaydedildi: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Dilara/Deney 1/Sonuçlar/VGG16_HOG_GIST_RBF_SVM_Mouth/20260807_1410_mouth_vgg16scratch_hog_gist_rbf_svm_seed42/artifacts/feature_cache/test_features.npz

FEATURE EXTRACTION COMPLETED
Train CNN / HOG / GIST: (2310, 512) (2310, 3780) (2310, 512)
Validation CNN / HOG / GIST: (287, 512) (287, 3780) (287, 512)
Test CNN / HOG / GIST: (292, 512) (292, 3780) (292, 512)
Ham birleşik özellik boyutu: 4804
Toplam süre: 3.62 dakika
Cache klasörü: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Dilara/Deney 1/Sonuçlar/VGG16_HOG_GIST_RBF_SVM_Mouth/20260807_1410_mouth_vgg16scratch_hog_gist_rbf_svm_seed42/artifacts/feature_cache

✅ VGG16 özellikleri çıkarıldı.
✅ HOG özellikleri çıkarıldı.
✅ GIST/Gabor özellikleri çıkarıldı.
✅ Özellikler split bazında Drive'a kaydedildi.
✅ Model seçimi için test etiketi kullanılmadı.


In [26]:
# ============================================================
# HÜCRE 18 — ÖZELLİK BİRLEŞTİRME + PCA + RBF-SVM SEÇİMİ
# TRAIN-ONLY PREPROCESSING / VALIDATION-ONLY MODEL SELECTION
# ============================================================

import os
import json
import time
import joblib
import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.preprocessing import (
    normalize,
    StandardScaler
)
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    precision_recall_curve
)


# ------------------------------------------------------------
# 1. Ayarlar
# ------------------------------------------------------------

PCA_COMPONENTS = int(
    getattr(CONFIG, "pca_components", 256)
)

SVM_C_VALUES = [0.1, 1.0, 10.0, 100.0]
SVM_GAMMA_VALUES = ["scale", 1e-3, 1e-4]

MODEL_ARTIFACT_DIR = (
    Path(OUTPUT_DIRS["artifacts"])
    / "svm_pipeline"
)

MODEL_ARTIFACT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

GRID_RESULTS_PATH = (
    Path(OUTPUT_DIRS["metrics"])
    / "svm_validation_grid_results.csv"
)

SELECTION_SUMMARY_PATH = (
    Path(OUTPUT_DIRS["metrics"])
    / "svm_validation_selection.json"
)

PROCESSED_FEATURES_PATH = (
    Path(OUTPUT_DIRS["artifacts"])
    / "pca_features.npz"
)


# ------------------------------------------------------------
# 2. Etiketler
# ------------------------------------------------------------

y_train = train_features["labels"].astype(
    np.int64
)

y_validation = validation_features[
    "labels"
].astype(np.int64)

# Test etiketi bu hücrede metrik hesabında kullanılmayacak.
assert set(np.unique(y_train)).issubset({0, 1})
assert set(np.unique(y_validation)).issubset({0, 1})


# ------------------------------------------------------------
# 3. Her özellik grubuna ayrı L2 normalizasyon
# ------------------------------------------------------------

def l2_normalize_feature_groups(feature_data):
    return {
        "cnn": normalize(
            feature_data["cnn"],
            norm="l2",
            axis=1
        ).astype(np.float32),

        "hog": normalize(
            feature_data["hog"],
            norm="l2",
            axis=1
        ).astype(np.float32),

        "gist": normalize(
            feature_data["gist"],
            norm="l2",
            axis=1
        ).astype(np.float32)
    }


train_l2 = l2_normalize_feature_groups(
    train_features
)

validation_l2 = l2_normalize_feature_groups(
    validation_features
)

test_l2 = l2_normalize_feature_groups(
    test_features
)


# ------------------------------------------------------------
# 4. StandardScaler — yalnızca train üzerinde fit
# ------------------------------------------------------------

cnn_scaler = StandardScaler()
hog_scaler = StandardScaler()
gist_scaler = StandardScaler()

train_cnn_scaled = cnn_scaler.fit_transform(
    train_l2["cnn"]
).astype(np.float32)

train_hog_scaled = hog_scaler.fit_transform(
    train_l2["hog"]
).astype(np.float32)

train_gist_scaled = gist_scaler.fit_transform(
    train_l2["gist"]
).astype(np.float32)


# Validation ve test yalnızca transform edilir.
validation_cnn_scaled = cnn_scaler.transform(
    validation_l2["cnn"]
).astype(np.float32)

validation_hog_scaled = hog_scaler.transform(
    validation_l2["hog"]
).astype(np.float32)

validation_gist_scaled = gist_scaler.transform(
    validation_l2["gist"]
).astype(np.float32)

test_cnn_scaled = cnn_scaler.transform(
    test_l2["cnn"]
).astype(np.float32)

test_hog_scaled = hog_scaler.transform(
    test_l2["hog"]
).astype(np.float32)

test_gist_scaled = gist_scaler.transform(
    test_l2["gist"]
).astype(np.float32)


# ------------------------------------------------------------
# 5. Özellik birleştirme
# ------------------------------------------------------------

X_train_combined = np.concatenate(
    [
        train_cnn_scaled,
        train_hog_scaled,
        train_gist_scaled
    ],
    axis=1
).astype(np.float32)

X_validation_combined = np.concatenate(
    [
        validation_cnn_scaled,
        validation_hog_scaled,
        validation_gist_scaled
    ],
    axis=1
).astype(np.float32)

X_test_combined = np.concatenate(
    [
        test_cnn_scaled,
        test_hog_scaled,
        test_gist_scaled
    ],
    axis=1
).astype(np.float32)

assert X_train_combined.shape == (2310, 4804)
assert X_validation_combined.shape == (287, 4804)
assert X_test_combined.shape == (292, 4804)


# ------------------------------------------------------------
# 6. PCA — yalnızca train üzerinde fit
# ------------------------------------------------------------

maximum_valid_components = min(
    X_train_combined.shape[0] - 1,
    X_train_combined.shape[1]
)

actual_pca_components = min(
    PCA_COMPONENTS,
    maximum_valid_components
)

pca = PCA(
    n_components=actual_pca_components,
    svd_solver="randomized",
    random_state=SEED
)

print("PCA train verisi üzerinde eğitiliyor...")

pca_start = time.time()

X_train_pca = pca.fit_transform(
    X_train_combined
).astype(np.float32)

X_validation_pca = pca.transform(
    X_validation_combined
).astype(np.float32)

X_test_pca = pca.transform(
    X_test_combined
).astype(np.float32)

pca_minutes = (
    time.time() - pca_start
) / 60.0

explained_variance = float(
    pca.explained_variance_ratio_.sum()
)

assert X_train_pca.shape == (
    2310,
    actual_pca_components
)

assert X_validation_pca.shape == (
    287,
    actual_pca_components
)

assert X_test_pca.shape == (
    292,
    actual_pca_components
)

print(
    f"PCA tamamlandı: {pca_minutes:.2f} dakika"
)
print(
    "Açıklanan toplam varyans:",
    round(explained_variance, 6)
)


# ------------------------------------------------------------
# 7. Validation üzerinde en iyi karar eşiğini bul
# ------------------------------------------------------------

def find_best_f1_threshold(
    true_labels,
    positive_probabilities
):
    precision_values, recall_values, thresholds = (
        precision_recall_curve(
            true_labels,
            positive_probabilities
        )
    )

    if len(thresholds) == 0:
        return 0.5, 0.0

    precision_for_thresholds = (
        precision_values[:-1]
    )

    recall_for_thresholds = (
        recall_values[:-1]
    )

    denominators = (
        precision_for_thresholds
        + recall_for_thresholds
    )

    f1_values = np.divide(
        2
        * precision_for_thresholds
        * recall_for_thresholds,
        denominators,
        out=np.zeros_like(
            denominators,
            dtype=np.float64
        ),
        where=denominators > 0
    )

    best_index = int(
        np.argmax(f1_values)
    )

    return (
        float(thresholds[best_index]),
        float(f1_values[best_index])
    )


# ------------------------------------------------------------
# 8. RBF-SVM grid araması
# ------------------------------------------------------------

grid_results = []
trained_candidates = {}

print("\nRBF-SVM VALIDATION GRID SEARCH")
print("=" * 64)

grid_start = time.time()

for c_value in SVM_C_VALUES:
    for gamma_value in SVM_GAMMA_VALUES:
        candidate_start = time.time()

        candidate_model = SVC(
            kernel="rbf",
            C=c_value,
            gamma=gamma_value,
            class_weight="balanced",
            probability=True,
            random_state=SEED,
            cache_size=2048
        )

        candidate_model.fit(
            X_train_pca,
            y_train
        )

        validation_probabilities = (
            candidate_model.predict_proba(
                X_validation_pca
            )[:, 1]
        )

        best_threshold, threshold_f1 = (
            find_best_f1_threshold(
                y_validation,
                validation_probabilities
            )
        )

        validation_predictions = (
            validation_probabilities
            >= best_threshold
        ).astype(np.int64)

        validation_auc = roc_auc_score(
            y_validation,
            validation_probabilities
        )

        result_row = {
            "C": float(c_value),
            "gamma": str(gamma_value),
            "threshold": float(best_threshold),
            "validation_accuracy": float(
                accuracy_score(
                    y_validation,
                    validation_predictions
                )
            ),
            "validation_precision": float(
                precision_score(
                    y_validation,
                    validation_predictions,
                    zero_division=0
                )
            ),
            "validation_recall": float(
                recall_score(
                    y_validation,
                    validation_predictions,
                    zero_division=0
                )
            ),
            "validation_f1": float(
                f1_score(
                    y_validation,
                    validation_predictions,
                    zero_division=0
                )
            ),
            "validation_roc_auc": float(
                validation_auc
            ),
            "fit_seconds": float(
                time.time() - candidate_start
            )
        }

        grid_results.append(result_row)

        model_key = (
            float(c_value),
            str(gamma_value)
        )

        trained_candidates[model_key] = (
            candidate_model
        )

        print(
            f"C={c_value:<6} | "
            f"gamma={str(gamma_value):<6} | "
            f"F1={result_row['validation_f1']:.4f} | "
            f"AUC={result_row['validation_roc_auc']:.4f} | "
            f"threshold={best_threshold:.4f}"
        )


# ------------------------------------------------------------
# 9. En iyi modeli seç
# Öncelik Validation F1, eşitlikte ROC-AUC ve accuracy
# ------------------------------------------------------------

grid_results_df = pd.DataFrame(
    grid_results
)

grid_results_df = grid_results_df.sort_values(
    by=[
        "validation_f1",
        "validation_roc_auc",
        "validation_accuracy"
    ],
    ascending=[False, False, False]
).reset_index(drop=True)

best_result = grid_results_df.iloc[0]

best_c = float(best_result["C"])
best_gamma_string = str(
    best_result["gamma"]
)

best_threshold = float(
    best_result["threshold"]
)

best_model_key = (
    best_c,
    best_gamma_string
)

best_svm = trained_candidates[
    best_model_key
]

assert best_svm is not None


# ------------------------------------------------------------
# 10. Grid sonuçlarını atomik kaydet
# ------------------------------------------------------------

grid_temp_path = GRID_RESULTS_PATH.with_suffix(
    ".csv.tmp"
)

grid_results_df.to_csv(
    grid_temp_path,
    index=False
)

os.replace(
    grid_temp_path,
    GRID_RESULTS_PATH
)


# ------------------------------------------------------------
# 11. Model bileşenlerini atomik kaydet
# ------------------------------------------------------------

def atomic_joblib_dump(
    object_to_save,
    destination
):
    destination = Path(destination)

    temporary_path = destination.with_suffix(
        destination.suffix + ".tmp"
    )

    joblib.dump(
        object_to_save,
        temporary_path
    )

    os.replace(
        temporary_path,
        destination
    )


atomic_joblib_dump(
    cnn_scaler,
    MODEL_ARTIFACT_DIR / "cnn_scaler.joblib"
)

atomic_joblib_dump(
    hog_scaler,
    MODEL_ARTIFACT_DIR / "hog_scaler.joblib"
)

atomic_joblib_dump(
    gist_scaler,
    MODEL_ARTIFACT_DIR / "gist_scaler.joblib"
)

atomic_joblib_dump(
    pca,
    MODEL_ARTIFACT_DIR / "pca.joblib"
)

atomic_joblib_dump(
    best_svm,
    MODEL_ARTIFACT_DIR / "rbf_svm.joblib"
)


# ------------------------------------------------------------
# 12. PCA özelliklerini atomik kaydet
# ------------------------------------------------------------

atomic_save_npz(
    PROCESSED_FEATURES_PATH,
    X_train=X_train_pca,
    y_train=y_train,
    X_validation=X_validation_pca,
    y_validation=y_validation,
    X_test=X_test_pca,
    train_file_names=train_features[
        "file_names"
    ],
    validation_file_names=validation_features[
        "file_names"
    ],
    test_file_names=test_features[
        "file_names"
    ],
    train_source_videos=train_features[
        "source_videos"
    ],
    validation_source_videos=validation_features[
        "source_videos"
    ],
    test_source_videos=test_features[
        "source_videos"
    ]
)


# ------------------------------------------------------------
# 13. Model seçim özetini kaydet
# ------------------------------------------------------------

grid_minutes = (
    time.time() - grid_start
) / 60.0

selection_summary = {
    "run_id": str(RUN_ID),
    "pipeline": (
        "VGG16 + HOG + GIST + PCA + RBF-SVM"
    ),
    "raw_feature_dimension": 4804,
    "pca_components": int(
        actual_pca_components
    ),
    "pca_explained_variance_ratio": (
        explained_variance
    ),
    "preprocessing_fit_scope": (
        "train_only"
    ),
    "model_selection_scope": (
        "validation_only"
    ),
    "selection_order": [
        "validation_f1",
        "validation_roc_auc",
        "validation_accuracy"
    ],
    "best_C": best_c,
    "best_gamma": best_gamma_string,
    "best_threshold": best_threshold,
    "best_validation_accuracy": float(
        best_result["validation_accuracy"]
    ),
    "best_validation_precision": float(
        best_result["validation_precision"]
    ),
    "best_validation_recall": float(
        best_result["validation_recall"]
    ),
    "best_validation_f1": float(
        best_result["validation_f1"]
    ),
    "best_validation_roc_auc": float(
        best_result["validation_roc_auc"]
    ),
    "pca_minutes": pca_minutes,
    "grid_search_minutes": grid_minutes,
    "test_metrics_calculated": False,
    "seed": SEED
}

summary_temp_path = (
    SELECTION_SUMMARY_PATH.with_suffix(
        ".json.tmp"
    )
)

with open(
    summary_temp_path,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        selection_summary,
        file,
        ensure_ascii=False,
        indent=2
    )

os.replace(
    summary_temp_path,
    SELECTION_SUMMARY_PATH
)


# ------------------------------------------------------------
# 14. Sonuç
# ------------------------------------------------------------

print("\nMODEL SELECTION COMPLETED")
print("=" * 64)

print("Ham birleşik boyut:", 4804)
print("PCA boyutu:", actual_pca_components)

print(
    "PCA açıklanan varyans:",
    round(explained_variance, 6)
)

print("En iyi C:", best_c)
print("En iyi gamma:", best_gamma_string)

print(
    "Validation karar eşiği:",
    round(best_threshold, 6)
)

print(
    "Validation Accuracy:",
    round(
        float(best_result["validation_accuracy"]),
        6
    )
)

print(
    "Validation F1:",
    round(
        float(best_result["validation_f1"]),
        6
    )
)

print(
    "Validation ROC-AUC:",
    round(
        float(best_result["validation_roc_auc"]),
        6
    )
)

print("Grid sonuçları:", GRID_RESULTS_PATH)
print("Model klasörü:", MODEL_ARTIFACT_DIR)

print("\n✅ L2 normalizasyon uygulandı.")
print("✅ Scaler ve PCA yalnızca train üzerinde fit edildi.")
print("✅ SVM seçimi yalnızca validation ile yapıldı.")
print("✅ Karar eşiği yalnızca validation ile belirlendi.")
print("✅ Test metriği henüz hesaplanmadı.")

PCA train verisi üzerinde eğitiliyor...
PCA tamamlandı: 0.07 dakika
Açıklanan toplam varyans: 0.860805

RBF-SVM VALIDATION GRID SEARCH
C=0.1    | gamma=scale  | F1=0.6521 | AUC=0.6365 | threshold=0.2818
C=0.1    | gamma=0.001  | F1=0.6614 | AUC=0.6362 | threshold=0.4868
C=0.1    | gamma=0.0001 | F1=0.6493 | AUC=0.6244 | threshold=0.2418
C=1.0    | gamma=scale  | F1=0.6824 | AUC=0.7098 | threshold=0.4768
C=1.0    | gamma=0.001  | F1=0.6796 | AUC=0.6853 | threshold=0.3779
C=1.0    | gamma=0.0001 | F1=0.6833 | AUC=0.7016 | threshold=0.3768
C=10.0   | gamma=scale  | F1=0.6854 | AUC=0.6766 | threshold=0.4448
C=10.0   | gamma=0.001  | F1=0.6649 | AUC=0.6616 | threshold=0.3475
C=10.0   | gamma=0.0001 | F1=0.6770 | AUC=0.6856 | threshold=0.3422
C=100.0  | gamma=scale  | F1=0.6702 | AUC=0.6359 | threshold=0.3674
C=100.0  | gamma=0.001  | F1=0.6649 | AUC=0.6616 | threshold=0.3475
C=100.0  | gamma=0.0001 | F1=0.6648 | AUC=0.6382 | threshold=0.4126

MODEL SELECTION COMPLETED
Ham birleşik boyut: 48

In [28]:
# ============================================================
# HÜCRE 19A — FINAL TEST TAHMİNLERİ VE METRİKLER
# ============================================================

import os
import json
import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    matthews_corrcoef,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)


# ------------------------------------------------------------
# 1. Çıktı yolları
# ------------------------------------------------------------

FRAME_PREDICTIONS_PATH = (
    Path(OUTPUT_DIRS["predictions"])
    / "frame_level_test_predictions.csv"
)

VIDEO_PREDICTIONS_PATH = (
    Path(OUTPUT_DIRS["predictions"])
    / "video_level_test_predictions.csv"
)

FRAME_METRICS_PATH = (
    Path(OUTPUT_DIRS["metrics"])
    / "frame_level_test_metrics.json"
)

VIDEO_METRICS_PATH = (
    Path(OUTPUT_DIRS["metrics"])
    / "video_level_test_metrics.json"
)


# ------------------------------------------------------------
# 2. Atomik kayıt fonksiyonları
# ------------------------------------------------------------

def atomic_dataframe_save(dataframe, destination):
    destination = Path(destination)
    temporary_path = destination.with_suffix(
        ".csv.tmp"
    )

    dataframe.to_csv(
        temporary_path,
        index=False
    )

    os.replace(
        temporary_path,
        destination
    )


def atomic_json_save(data, destination):
    destination = Path(destination)
    temporary_path = destination.with_suffix(
        ".json.tmp"
    )

    with open(
        temporary_path,
        "w",
        encoding="utf-8"
    ) as file:
        json.dump(
            data,
            file,
            ensure_ascii=False,
            indent=2
        )

    os.replace(
        temporary_path,
        destination
    )


# ------------------------------------------------------------
# 3. Binary metrik fonksiyonu
# ------------------------------------------------------------

def calculate_binary_metrics(
    true_labels,
    predicted_labels,
    positive_probabilities
):
    matrix = confusion_matrix(
        true_labels,
        predicted_labels,
        labels=[0, 1]
    )

    tn, fp, fn, tp = matrix.ravel()

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else 0.0
    )

    npv = (
        tn / (tn + fn)
        if (tn + fn) > 0
        else 0.0
    )

    return {
        "sample_count": int(len(true_labels)),
        "accuracy": float(
            accuracy_score(
                true_labels,
                predicted_labels
            )
        ),
        "balanced_accuracy": float(
            balanced_accuracy_score(
                true_labels,
                predicted_labels
            )
        ),
        "precision": float(
            precision_score(
                true_labels,
                predicted_labels,
                zero_division=0
            )
        ),
        "recall_sensitivity": float(
            recall_score(
                true_labels,
                predicted_labels,
                zero_division=0
            )
        ),
        "specificity": float(specificity),
        "negative_predictive_value": float(npv),
        "f1": float(
            f1_score(
                true_labels,
                predicted_labels,
                zero_division=0
            )
        ),
        "matthews_correlation_coefficient": float(
            matthews_corrcoef(
                true_labels,
                predicted_labels
            )
        ),
        "roc_auc": float(
            roc_auc_score(
                true_labels,
                positive_probabilities
            )
        ),
        "pr_auc_average_precision": float(
            average_precision_score(
                true_labels,
                positive_probabilities
            )
        ),
        "true_negative": int(tn),
        "false_positive": int(fp),
        "false_negative": int(fn),
        "true_positive": int(tp)
    }


# ------------------------------------------------------------
# 4. Frame-level final test
# ------------------------------------------------------------

y_test = test_features["labels"].astype(
    np.int64
)

test_fake_probabilities = (
    best_svm.predict_proba(
        X_test_pca
    )[:, 1]
)

test_frame_predictions = (
    test_fake_probabilities >= best_threshold
).astype(np.int64)

assert len(y_test) == 292
assert np.isfinite(
    test_fake_probabilities
).all()

frame_metrics = calculate_binary_metrics(
    y_test,
    test_frame_predictions,
    test_fake_probabilities
)

frame_predictions_df = pd.DataFrame({
    "run_id": str(RUN_ID),
    "file_name": test_features["file_names"],
    "source_video": test_features["source_videos"],
    "image_path": test_features["image_paths"],
    "true_label_id": y_test,
    "true_label": np.where(
        y_test == 1,
        "fake",
        "real"
    ),
    "fake_probability": test_fake_probabilities,
    "decision_threshold": float(best_threshold),
    "predicted_label_id": test_frame_predictions,
    "predicted_label": np.where(
        test_frame_predictions == 1,
        "fake",
        "real"
    ),
    "correct": (
        test_frame_predictions == y_test
    )
})


# ------------------------------------------------------------
# 5. Source-video-level birleştirme
# ------------------------------------------------------------

label_consistency = (
    frame_predictions_df
    .groupby("source_video")["true_label_id"]
    .nunique()
)

assert label_consistency.eq(1).all()

video_predictions_df = (
    frame_predictions_df
    .groupby("source_video", as_index=False)
    .agg(
        true_label_id=(
            "true_label_id",
            "first"
        ),
        fake_probability=(
            "fake_probability",
            "mean"
        ),
        frame_count=(
            "file_name",
            "count"
        )
    )
)

video_predictions_df[
    "decision_threshold"
] = float(best_threshold)

video_predictions_df[
    "predicted_label_id"
] = (
    video_predictions_df["fake_probability"]
    >= best_threshold
).astype(np.int64)

video_predictions_df["true_label"] = np.where(
    video_predictions_df["true_label_id"] == 1,
    "fake",
    "real"
)

video_predictions_df[
    "predicted_label"
] = np.where(
    video_predictions_df[
        "predicted_label_id"
    ] == 1,
    "fake",
    "real"
)

video_predictions_df["correct"] = (
    video_predictions_df["true_label_id"]
    == video_predictions_df[
        "predicted_label_id"
    ]
)

video_predictions_df.insert(
    0,
    "run_id",
    str(RUN_ID)
)

video_true_labels = video_predictions_df[
    "true_label_id"
].to_numpy(dtype=np.int64)

video_fake_probabilities = video_predictions_df[
    "fake_probability"
].to_numpy(dtype=np.float64)

video_predicted_labels = video_predictions_df[
    "predicted_label_id"
].to_numpy(dtype=np.int64)

video_metrics = calculate_binary_metrics(
    video_true_labels,
    video_predicted_labels,
    video_fake_probabilities
)


# ------------------------------------------------------------
# 6. Tahminleri kaydet
# ------------------------------------------------------------

atomic_dataframe_save(
    frame_predictions_df,
    FRAME_PREDICTIONS_PATH
)

atomic_dataframe_save(
    video_predictions_df,
    VIDEO_PREDICTIONS_PATH
)


# ------------------------------------------------------------
# 7. Metrikleri kaydet
# ------------------------------------------------------------

frame_metrics_record = {
    "run_id": str(RUN_ID),
    "evaluation_level": "frame",
    "model": (
        "VGG16 + HOG + GIST + PCA + RBF-SVM"
    ),
    "threshold_source": "validation_only",
    "decision_threshold": float(best_threshold),
    **frame_metrics
}

video_metrics_record = {
    "run_id": str(RUN_ID),
    "evaluation_level": "source_video",
    "aggregation": (
        "mean frame-level fake probability"
    ),
    "model": (
        "VGG16 + HOG + GIST + PCA + RBF-SVM"
    ),
    "threshold_source": "validation_only",
    "decision_threshold": float(best_threshold),
    **video_metrics
}

atomic_json_save(
    frame_metrics_record,
    FRAME_METRICS_PATH
)

atomic_json_save(
    video_metrics_record,
    VIDEO_METRICS_PATH
)


# ------------------------------------------------------------
# 8. Sonuçları göster
# ------------------------------------------------------------

print("FINAL TEST METRICS")
print("=" * 62)

print("\nFRAME-LEVEL")
print("-" * 62)
print("Frame count:", frame_metrics["sample_count"])
print("Accuracy:", round(frame_metrics["accuracy"], 6))
print("Precision:", round(frame_metrics["precision"], 6))
print(
    "Recall:",
    round(frame_metrics["recall_sensitivity"], 6)
)
print(
    "Specificity:",
    round(frame_metrics["specificity"], 6)
)
print("F1:", round(frame_metrics["f1"], 6))
print("ROC-AUC:", round(frame_metrics["roc_auc"], 6))
print(
    "PR-AUC:",
    round(
        frame_metrics["pr_auc_average_precision"],
        6
    )
)
print(
    "TN / FP / FN / TP:",
    frame_metrics["true_negative"],
    frame_metrics["false_positive"],
    frame_metrics["false_negative"],
    frame_metrics["true_positive"]
)

print("\nVIDEO-LEVEL")
print("-" * 62)
print("Video count:", video_metrics["sample_count"])
print("Accuracy:", round(video_metrics["accuracy"], 6))
print("Precision:", round(video_metrics["precision"], 6))
print(
    "Recall:",
    round(video_metrics["recall_sensitivity"], 6)
)
print(
    "Specificity:",
    round(video_metrics["specificity"], 6)
)
print("F1:", round(video_metrics["f1"], 6))
print("ROC-AUC:", round(video_metrics["roc_auc"], 6))
print(
    "PR-AUC:",
    round(
        video_metrics["pr_auc_average_precision"],
        6
    )
)

print("\n✅ Final test metrikleri hesaplandı.")
print("✅ Frame tahminleri Drive'a kaydedildi.")
print("✅ Video tahminleri Drive'a kaydedildi.")
print("✅ Validation eşiği değiştirilmeden kullanıldı.")

FINAL TEST METRICS

FRAME-LEVEL
--------------------------------------------------------------
Frame count: 292
Accuracy: 0.575342
Precision: 0.560847
Recall: 0.721088
Specificity: 0.427586
F1: 0.630952
ROC-AUC: 0.672156
PR-AUC: 0.690503
TN / FP / FN / TP: 62 83 41 106

VIDEO-LEVEL
--------------------------------------------------------------
Video count: 196
Accuracy: 0.622449
Precision: 0.700787
Recall: 0.712
Specificity: 0.464789
F1: 0.706349
ROC-AUC: 0.698197
PR-AUC: 0.826189

✅ Final test metrikleri hesaplandı.
✅ Frame tahminleri Drive'a kaydedildi.
✅ Video tahminleri Drive'a kaydedildi.
✅ Validation eşiği değiştirilmeden kullanıldı.


In [29]:
# ============================================================
# HÜCRE 19B — FINAL TEST GRAFİKLERİ VE AUDIT KAYDI
# ============================================================

import os
import json
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from PIL import Image
from sklearn.metrics import (
    confusion_matrix,
    roc_curve,
    precision_recall_curve
)


# ------------------------------------------------------------
# 1. Çıktı yolları
# ------------------------------------------------------------

FINAL_FIGURE_PATH = (
    Path(OUTPUT_DIRS["figures"])
    / "final_test_evaluation.png"
)

TEST_AUDIT_PATH = (
    Path(OUTPUT_DIRS["artifacts"])
    / "final_test_evaluation_audit.json"
)


# ------------------------------------------------------------
# 2. Eğri ve confusion matrix verileri
# ------------------------------------------------------------

frame_matrix = confusion_matrix(
    y_test,
    test_frame_predictions,
    labels=[0, 1]
)

video_matrix = confusion_matrix(
    video_true_labels,
    video_predicted_labels,
    labels=[0, 1]
)

frame_fpr, frame_tpr, _ = roc_curve(
    y_test,
    test_fake_probabilities
)

video_fpr, video_tpr, _ = roc_curve(
    video_true_labels,
    video_fake_probabilities
)

frame_precision_curve, frame_recall_curve, _ = (
    precision_recall_curve(
        y_test,
        test_fake_probabilities
    )
)

video_precision_curve, video_recall_curve, _ = (
    precision_recall_curve(
        video_true_labels,
        video_fake_probabilities
    )
)


# ------------------------------------------------------------
# 3. İngilizce final grafik
# ------------------------------------------------------------

plt.style.use("seaborn-v0_8-whitegrid")

figure, axes = plt.subplots(
    2,
    2,
    figsize=(13, 10)
)

sns.heatmap(
    frame_matrix,
    annot=True,
    fmt="d",
    cmap="Blues",
    cbar=False,
    xticklabels=["Real", "Fake"],
    yticklabels=["Real", "Fake"],
    ax=axes[0, 0]
)

axes[0, 0].set_title(
    "Frame-Level Confusion Matrix"
)
axes[0, 0].set_xlabel("Predicted Label")
axes[0, 0].set_ylabel("True Label")


sns.heatmap(
    video_matrix,
    annot=True,
    fmt="d",
    cmap="Oranges",
    cbar=False,
    xticklabels=["Real", "Fake"],
    yticklabels=["Real", "Fake"],
    ax=axes[0, 1]
)

axes[0, 1].set_title(
    "Video-Level Confusion Matrix"
)
axes[0, 1].set_xlabel("Predicted Label")
axes[0, 1].set_ylabel("True Label")


axes[1, 0].plot(
    frame_fpr,
    frame_tpr,
    linewidth=2.2,
    color="#1f77b4",
    label=(
        f"Frame-Level "
        f"(AUC={frame_metrics['roc_auc']:.3f})"
    )
)

axes[1, 0].plot(
    video_fpr,
    video_tpr,
    linewidth=2.2,
    color="#ff7f0e",
    label=(
        f"Video-Level "
        f"(AUC={video_metrics['roc_auc']:.3f})"
    )
)

axes[1, 0].plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    color="gray",
    label="Random Classifier"
)

axes[1, 0].set_title("ROC Curves")
axes[1, 0].set_xlabel("False Positive Rate")
axes[1, 0].set_ylabel("True Positive Rate")
axes[1, 0].set_xlim(0, 1)
axes[1, 0].set_ylim(0, 1.02)
axes[1, 0].legend(loc="lower right")


axes[1, 1].plot(
    frame_recall_curve,
    frame_precision_curve,
    linewidth=2.2,
    color="#1f77b4",
    label=(
        f"Frame-Level "
        f"(AP={frame_metrics['pr_auc_average_precision']:.3f})"
    )
)

axes[1, 1].plot(
    video_recall_curve,
    video_precision_curve,
    linewidth=2.2,
    color="#ff7f0e",
    label=(
        f"Video-Level "
        f"(AP={video_metrics['pr_auc_average_precision']:.3f})"
    )
)

axes[1, 1].set_title(
    "Precision-Recall Curves"
)
axes[1, 1].set_xlabel("Recall")
axes[1, 1].set_ylabel("Precision")
axes[1, 1].set_xlim(0, 1)
axes[1, 1].set_ylim(0, 1.02)
axes[1, 1].legend(loc="lower left")


figure.suptitle(
    "Final Test Evaluation — Mouth ROI Deepfake Detection",
    fontsize=15,
    fontweight="bold"
)

figure.tight_layout(
    rect=[0, 0, 1, 0.96]
)

temporary_figure_path = (
    FINAL_FIGURE_PATH.with_suffix(
        ".png.tmp"
    )
)

figure.savefig(
    temporary_figure_path,
    format="png",
    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)

plt.close(figure)

os.replace(
    temporary_figure_path,
    FINAL_FIGURE_PATH
)


# ------------------------------------------------------------
# 4. Grafik kalite kontrolü
# ------------------------------------------------------------

with Image.open(FINAL_FIGURE_PATH) as image:
    final_figure_width = image.width
    final_figure_height = image.height

assert min(
    final_figure_width,
    final_figure_height
) >= 600


# ------------------------------------------------------------
# 5. Final test audit kaydı
# ------------------------------------------------------------

test_audit_record = {
    "run_id": str(RUN_ID),
    "model": (
        "VGG16 + HOG + GIST + PCA + RBF-SVM"
    ),
    "test_evaluation_stage": (
        "after preprocessing, hyperparameters "
        "and threshold were frozen"
    ),
    "test_frame_count": int(len(y_test)),
    "test_source_video_count": int(
        len(video_predictions_df)
    ),
    "decision_threshold": float(
        best_threshold
    ),
    "threshold_selected_using": (
        "validation_only"
    ),
    "test_used_for_preprocessing_fit": False,
    "test_used_for_model_selection": False,
    "test_used_for_threshold_selection": False,
    "frame_predictions": str(
        FRAME_PREDICTIONS_PATH
    ),
    "video_predictions": str(
        VIDEO_PREDICTIONS_PATH
    ),
    "frame_metrics": str(
        FRAME_METRICS_PATH
    ),
    "video_metrics": str(
        VIDEO_METRICS_PATH
    ),
    "final_figure": str(
        FINAL_FIGURE_PATH
    ),
    "figure_dpi": 600,
    "figure_size_pixels": [
        int(final_figure_width),
        int(final_figure_height)
    ]
}

atomic_json_save(
    test_audit_record,
    TEST_AUDIT_PATH
)


# ------------------------------------------------------------
# 6. Sonuç
# ------------------------------------------------------------

print("FINAL TEST ARTIFACTS")
print("=" * 62)

print("Final figure:", FINAL_FIGURE_PATH)

print(
    "Figure resolution:",
    f"{final_figure_width}x"
    f"{final_figure_height}"
)

print("Figure DPI: 600")
print("Test audit:", TEST_AUDIT_PATH)

print("\nFrame confusion matrix:")
print(frame_matrix)

print("\nVideo confusion matrix:")
print(video_matrix)

print("\n✅ Final test grafikleri kaydedildi.")
print("✅ Grafiklerin tamamı İngilizce.")
print("✅ Grafik 600 DPI kalite standardını karşılıyor.")
print("✅ Test değerlendirme audit kaydı oluşturuldu.")

FINAL TEST ARTIFACTS
Final figure: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Dilara/Deney 1/Sonuçlar/VGG16_HOG_GIST_RBF_SVM_Mouth/20260807_1410_mouth_vgg16scratch_hog_gist_rbf_svm_seed42/figures/final_test_evaluation.png
Figure resolution: 7734x5898
Figure DPI: 600
Test audit: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Dilara/Deney 1/Sonuçlar/VGG16_HOG_GIST_RBF_SVM_Mouth/20260807_1410_mouth_vgg16scratch_hog_gist_rbf_svm_seed42/artifacts/final_test_evaluation_audit.json

Frame confusion matrix:
[[ 62  83]
 [ 41 106]]

Video confusion matrix:
[[33 38]
 [36 89]]

✅ Final test grafikleri kaydedildi.
✅ Grafiklerin tamamı İngilizce.
✅ Grafik 600 DPI kalite standardını karşılıyor.
✅ Test değerlendirme audit kaydı oluşturuldu.


In [30]:
# ============================================================
# HÜCRE 20 — NİHAİ DENEY AUDITİ VE DOSYA MANİFESTOSU
# ============================================================

import os
import json
import hashlib
import numpy as np
import pandas as pd

from pathlib import Path
from tqdm.auto import tqdm


# ------------------------------------------------------------
# 1. Yollar
# ------------------------------------------------------------

RUN_PATH = Path(RUN_DIR)

FINAL_SUMMARY_PATH = (
    Path(OUTPUT_DIRS["metrics"])
    / "final_experiment_summary.json"
)

FINAL_SUMMARY_TEXT_PATH = (
    Path(OUTPUT_DIRS["metrics"])
    / "final_experiment_summary.txt"
)

MANIFEST_PATH = (
    Path(OUTPUT_DIRS["artifacts"])
    / "experiment_file_manifest.csv"
)


# ------------------------------------------------------------
# 2. JSON dosyalarını oku
# ------------------------------------------------------------

def read_json(path):
    with open(
        path,
        "r",
        encoding="utf-8"
    ) as file:
        return json.load(file)


frame_metrics_saved = read_json(
    FRAME_METRICS_PATH
)

video_metrics_saved = read_json(
    VIDEO_METRICS_PATH
)

test_audit_saved = read_json(
    TEST_AUDIT_PATH
)

training_summary_saved = read_json(
    TRAINING_SUMMARY_PATH
)

selection_summary_saved = read_json(
    SELECTION_SUMMARY_PATH
)


# ------------------------------------------------------------
# 3. Zorunlu dosya kontrolü
# ------------------------------------------------------------

required_files = [
    RUN_PATH / "config_resolved.yaml",

    Path(OUTPUT_DIRS["artifacts"])
    / "metadata_full_audit.csv",

    Path(OUTPUT_DIRS["artifacts"])
    / "metadata_used.csv",

    Path(OUTPUT_DIRS["artifacts"])
    / "train_normalization.json",

    Path(OUTPUT_DIRS["artifacts"])
    / "data_pipeline_config.json",

    Path(OUTPUT_DIRS["artifacts"])
    / "vgg16_architecture.json",

    Path(OUTPUT_DIRS["checkpoints"])
    / "vgg16_best.pt",

    Path(OUTPUT_DIRS["checkpoints"])
    / "vgg16_last.pt",

    Path(OUTPUT_DIRS["logs"])
    / "vgg16_training_history.csv",

    Path(OUTPUT_DIRS["figures"])
    / "vgg16_training_curves.png",

    Path(OUTPUT_DIRS["artifacts"])
    / "feature_cache"
    / "train_features.npz",

    Path(OUTPUT_DIRS["artifacts"])
    / "feature_cache"
    / "validation_features.npz",

    Path(OUTPUT_DIRS["artifacts"])
    / "feature_cache"
    / "test_features.npz",

    Path(OUTPUT_DIRS["artifacts"])
    / "svm_pipeline"
    / "cnn_scaler.joblib",

    Path(OUTPUT_DIRS["artifacts"])
    / "svm_pipeline"
    / "hog_scaler.joblib",

    Path(OUTPUT_DIRS["artifacts"])
    / "svm_pipeline"
    / "gist_scaler.joblib",

    Path(OUTPUT_DIRS["artifacts"])
    / "svm_pipeline"
    / "pca.joblib",

    Path(OUTPUT_DIRS["artifacts"])
    / "svm_pipeline"
    / "rbf_svm.joblib",

    Path(OUTPUT_DIRS["metrics"])
    / "svm_validation_grid_results.csv",

    FRAME_PREDICTIONS_PATH,
    VIDEO_PREDICTIONS_PATH,
    FRAME_METRICS_PATH,
    VIDEO_METRICS_PATH,
    FINAL_FIGURE_PATH,
    TEST_AUDIT_PATH
]

missing_required_files = [
    str(path)
    for path in required_files
    if not Path(path).exists()
]


# ------------------------------------------------------------
# 4. Veri muhasebesi
# ------------------------------------------------------------

metadata_full_path = (
    Path(OUTPUT_DIRS["artifacts"])
    / "metadata_full_audit.csv"
)

metadata_used_path = (
    Path(OUTPUT_DIRS["artifacts"])
    / "metadata_used.csv"
)

metadata_full_saved = pd.read_csv(
    metadata_full_path
)

metadata_used_saved = pd.read_csv(
    metadata_used_path
)

frame_predictions_saved = pd.read_csv(
    FRAME_PREDICTIONS_PATH
)

video_predictions_saved = pd.read_csv(
    VIDEO_PREDICTIONS_PATH
)

accounting_check = {
    "input_total": int(
        len(metadata_full_saved)
    ),
    "success": int(
        len(metadata_used_saved)
    ),
    "skipped": int(
        len(metadata_full_saved)
        - len(metadata_used_saved)
    ),
    "error": 0
}

accounting_is_valid = (
    accounting_check["input_total"]
    == accounting_check["success"]
    + accounting_check["skipped"]
    + accounting_check["error"]
)


# ------------------------------------------------------------
# 5. Source-video split izolasyonu
# ------------------------------------------------------------

train_videos = set(
    metadata_used_saved.loc[
        metadata_used_saved["split"]
        .astype(str)
        .str.lower()
        .eq("train"),
        "source_video"
    ].astype(str)
)

validation_videos = set(
    metadata_used_saved.loc[
        metadata_used_saved["split"]
        .astype(str)
        .str.lower()
        .eq("val"),
        "source_video"
    ].astype(str)
)

test_videos = set(
    metadata_used_saved.loc[
        metadata_used_saved["split"]
        .astype(str)
        .str.lower()
        .eq("test"),
        "source_video"
    ].astype(str)
)

train_validation_overlap = (
    train_videos & validation_videos
)

train_test_overlap = (
    train_videos & test_videos
)

validation_test_overlap = (
    validation_videos & test_videos
)

source_video_isolation_valid = (
    len(train_validation_overlap) == 0
    and len(train_test_overlap) == 0
    and len(validation_test_overlap) == 0
)


# ------------------------------------------------------------
# 6. SHA-256 split izolasyonu
# ------------------------------------------------------------

train_hashes = set(
    metadata_used_saved.loc[
        metadata_used_saved["split"]
        .astype(str)
        .str.lower()
        .eq("train"),
        "sha256"
    ].dropna().astype(str)
)

validation_hashes = set(
    metadata_used_saved.loc[
        metadata_used_saved["split"]
        .astype(str)
        .str.lower()
        .eq("val"),
        "sha256"
    ].dropna().astype(str)
)

test_hashes = set(
    metadata_used_saved.loc[
        metadata_used_saved["split"]
        .astype(str)
        .str.lower()
        .eq("test"),
        "sha256"
    ].dropna().astype(str)
)

hash_train_validation_overlap = (
    train_hashes & validation_hashes
)

hash_train_test_overlap = (
    train_hashes & test_hashes
)

hash_validation_test_overlap = (
    validation_hashes & test_hashes
)

hash_isolation_valid = (
    len(hash_train_validation_overlap) == 0
    and len(hash_train_test_overlap) == 0
    and len(hash_validation_test_overlap) == 0
)


# ------------------------------------------------------------
# 7. Test protokol kontrolü
# ------------------------------------------------------------

prediction_count_valid = (
    len(frame_predictions_saved) == 292
    and len(video_predictions_saved) == 196
)

threshold_consistent = (
    np.isclose(
        frame_predictions_saved[
            "decision_threshold"
        ].iloc[0],
        selection_summary_saved[
            "best_threshold"
        ]
    )
    and np.isclose(
        video_predictions_saved[
            "decision_threshold"
        ].iloc[0],
        selection_summary_saved[
            "best_threshold"
        ]
    )
)

test_protocol_valid = all([
    test_audit_saved[
        "test_used_for_preprocessing_fit"
    ] is False,

    test_audit_saved[
        "test_used_for_model_selection"
    ] is False,

    test_audit_saved[
        "test_used_for_threshold_selection"
    ] is False,

    threshold_consistent,
    prediction_count_valid
])


# ------------------------------------------------------------
# 8. Ham veri ve sonuç dizini ayrımı
# ------------------------------------------------------------

raw_data_path = Path(DATA_ROOT)
run_path_string = str(RUN_PATH)
raw_path_string = str(raw_data_path)

output_separate_from_raw = (
    not run_path_string.startswith(
        raw_path_string + os.sep
    )
)


# ------------------------------------------------------------
# 9. Geçici dosya kontrolü
# ------------------------------------------------------------

temporary_files = [
    str(path)
    for path in RUN_PATH.rglob("*")
    if path.is_file()
    and (
        path.name.endswith(".tmp")
        or ".tmp." in path.name
    )
]


# ------------------------------------------------------------
# 10. Dosya SHA-256 manifestosu
# ------------------------------------------------------------

def sha256_file(path):
    digest = hashlib.sha256()

    with open(path, "rb") as file:
        while True:
            chunk = file.read(1024 * 1024)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


manifest_candidates = sorted([
    path
    for path in RUN_PATH.rglob("*")
    if path.is_file()
    and path != MANIFEST_PATH
    and not path.name.endswith(".tmp")
])

manifest_rows = []

for path in tqdm(
    manifest_candidates,
    desc="Creating file manifest"
):
    manifest_rows.append({
        "relative_path": str(
            path.relative_to(RUN_PATH)
        ),
        "size_bytes": int(
            path.stat().st_size
        ),
        "sha256": sha256_file(path)
    })

manifest_df = pd.DataFrame(
    manifest_rows
)

manifest_temp_path = (
    MANIFEST_PATH.with_suffix(".csv.tmp")
)

manifest_df.to_csv(
    manifest_temp_path,
    index=False
)

os.replace(
    manifest_temp_path,
    MANIFEST_PATH
)


# ------------------------------------------------------------
# 11. Genel audit kararı
# ------------------------------------------------------------

audit_checks = {
    "required_files_present": (
        len(missing_required_files) == 0
    ),
    "data_accounting_valid": accounting_is_valid,
    "source_video_isolation_valid": (
        source_video_isolation_valid
    ),
    "sha256_split_isolation_valid": (
        hash_isolation_valid
    ),
    "test_protocol_valid": test_protocol_valid,
    "output_separate_from_raw_data": (
        output_separate_from_raw
    ),
    "no_temporary_files_remaining": (
        len(temporary_files) == 0
    ),
    "frame_prediction_count_valid": (
        len(frame_predictions_saved) == 292
    ),
    "video_prediction_count_valid": (
        len(video_predictions_saved) == 196
    ),
    "figure_quality_valid": (
        int(
            test_audit_saved[
                "figure_dpi"
            ]
        ) >= 600
        and min(
            test_audit_saved[
                "figure_size_pixels"
            ]
        ) >= 600
    )
}

all_audit_checks_passed = all(
    audit_checks.values()
)


# ------------------------------------------------------------
# 12. Nihai deney özeti
# ------------------------------------------------------------

final_summary = {
    "run_id": str(RUN_ID),
    "run_directory": str(RUN_PATH),
    "notebook_name": (
        "VGG16_HOG_GIST_RBF_SVM_"
        "Mouth_Deepfake_Colab_FIXED.ipynb"
    ),
    "experiment": (
        "VGG16 from scratch + HOG + GIST "
        "+ PCA + RBF-SVM on mouth ROI"
    ),
    "seed": SEED,

    "data": {
        **accounting_check,
        "train": 2310,
        "validation": 287,
        "test": 292,
        "test_source_videos": 196
    },

    "best_vgg16": {
        "epoch": int(
            training_summary_saved[
                "best_epoch"
            ]
        ),
        "validation_f1": float(
            training_summary_saved[
                "best_validation_f1"
            ]
        )
    },

    "selected_svm": {
        "C": selection_summary_saved[
            "best_C"
        ],
        "gamma": selection_summary_saved[
            "best_gamma"
        ],
        "threshold": selection_summary_saved[
            "best_threshold"
        ],
        "validation_f1": (
            selection_summary_saved[
                "best_validation_f1"
            ]
        ),
        "validation_roc_auc": (
            selection_summary_saved[
                "best_validation_roc_auc"
            ]
        )
    },

    "frame_level_test": {
        "accuracy": frame_metrics_saved[
            "accuracy"
        ],
        "precision": frame_metrics_saved[
            "precision"
        ],
        "recall": frame_metrics_saved[
            "recall_sensitivity"
        ],
        "specificity": frame_metrics_saved[
            "specificity"
        ],
        "f1": frame_metrics_saved["f1"],
        "roc_auc": frame_metrics_saved[
            "roc_auc"
        ],
        "pr_auc": frame_metrics_saved[
            "pr_auc_average_precision"
        ]
    },

    "video_level_test": {
        "accuracy": video_metrics_saved[
            "accuracy"
        ],
        "precision": video_metrics_saved[
            "precision"
        ],
        "recall": video_metrics_saved[
            "recall_sensitivity"
        ],
        "specificity": video_metrics_saved[
            "specificity"
        ],
        "f1": video_metrics_saved["f1"],
        "roc_auc": video_metrics_saved[
            "roc_auc"
        ],
        "pr_auc": video_metrics_saved[
            "pr_auc_average_precision"
        ]
    },

    "split_isolation": {
        "train_validation_source_overlap": (
            len(train_validation_overlap)
        ),
        "train_test_source_overlap": (
            len(train_test_overlap)
        ),
        "validation_test_source_overlap": (
            len(validation_test_overlap)
        ),
        "train_validation_hash_overlap": (
            len(hash_train_validation_overlap)
        ),
        "train_test_hash_overlap": (
            len(hash_train_test_overlap)
        ),
        "validation_test_hash_overlap": (
            len(hash_validation_test_overlap)
        )
    },

    "audit_checks": audit_checks,
    "all_audit_checks_passed": (
        all_audit_checks_passed
    ),

    "missing_required_files": (
        missing_required_files
    ),
    "temporary_files": temporary_files,
    "manifest_file_count": int(
        len(manifest_df)
    ),
    "manifest_path": str(MANIFEST_PATH)
}

summary_temp_path = (
    FINAL_SUMMARY_PATH.with_suffix(
        ".json.tmp"
    )
)

with open(
    summary_temp_path,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        final_summary,
        file,
        ensure_ascii=False,
        indent=2
    )

os.replace(
    summary_temp_path,
    FINAL_SUMMARY_PATH
)


# ------------------------------------------------------------
# 13. Okunabilir metin özeti
# ------------------------------------------------------------

summary_lines = [
    "FINAL EXPERIMENT SUMMARY",
    "=" * 70,
    f"Run ID: {RUN_ID}",
    f"Run directory: {RUN_PATH}",
    "",
    "DATA ACCOUNTING",
    f"Input: {accounting_check['input_total']}",
    f"Success: {accounting_check['success']}",
    f"Skipped: {accounting_check['skipped']}",
    f"Error: {accounting_check['error']}",
    "",
    "FRAME-LEVEL TEST",
    (
        f"Accuracy: "
        f"{frame_metrics_saved['accuracy']:.6f}"
    ),
    (
        f"F1: "
        f"{frame_metrics_saved['f1']:.6f}"
    ),
    (
        f"ROC-AUC: "
        f"{frame_metrics_saved['roc_auc']:.6f}"
    ),
    "",
    "VIDEO-LEVEL TEST",
    (
        f"Accuracy: "
        f"{video_metrics_saved['accuracy']:.6f}"
    ),
    (
        f"F1: "
        f"{video_metrics_saved['f1']:.6f}"
    ),
    (
        f"ROC-AUC: "
        f"{video_metrics_saved['roc_auc']:.6f}"
    ),
    "",
    "AUDIT RESULT",
    (
        "PASS"
        if all_audit_checks_passed
        else "FAIL"
    )
]

summary_text = "\n".join(summary_lines)

summary_text_temp = (
    FINAL_SUMMARY_TEXT_PATH.with_suffix(
        ".txt.tmp"
    )
)

with open(
    summary_text_temp,
    "w",
    encoding="utf-8"
) as file:
    file.write(summary_text)

os.replace(
    summary_text_temp,
    FINAL_SUMMARY_TEXT_PATH
)


# ------------------------------------------------------------
# 14. Final çıktı
# ------------------------------------------------------------

print("\nFINAL EXPERIMENT AUDIT")
print("=" * 70)

for check_name, check_result in audit_checks.items():
    symbol = "✅" if check_result else "❌"
    print(
        f"{symbol} {check_name}: {check_result}"
    )

print("\nDATA ACCOUNTING")
print("-" * 70)
print(accounting_check)

print("\nSOURCE-VIDEO OVERLAPS")
print("-" * 70)
print(
    "Train-validation:",
    len(train_validation_overlap)
)
print(
    "Train-test:",
    len(train_test_overlap)
)
print(
    "Validation-test:",
    len(validation_test_overlap)
)

print("\nHASH OVERLAPS")
print("-" * 70)
print(
    "Train-validation:",
    len(hash_train_validation_overlap)
)
print(
    "Train-test:",
    len(hash_train_test_overlap)
)
print(
    "Validation-test:",
    len(hash_validation_test_overlap)
)

print("\nFINAL RESULT")
print("-" * 70)
print(
    "Audit sonucu:",
    (
        "✅ PASS"
        if all_audit_checks_passed
        else "❌ FAIL"
    )
)
print("Manifest dosya sayısı:", len(manifest_df))
print("Nihai JSON:", FINAL_SUMMARY_PATH)
print("Nihai metin:", FINAL_SUMMARY_TEXT_PATH)
print("Dosya manifestosu:", MANIFEST_PATH)

if not all_audit_checks_passed:
    print("\nEksik dosyalar:", missing_required_files)
    print("Geçici dosyalar:", temporary_files)

assert all_audit_checks_passed, (
    "Nihai audit kontrollerinden en az biri başarısız."
)

print("\n✅ Deney başarıyla tamamlandı.")
print("✅ Tüm sonuçlar Drive run klasöründe.")
print("✅ Source-video leakage bulunmadı.")
print("✅ Splitler arasında aynı görüntü hash'i bulunmadı.")
print("✅ Ham ağız verileri değiştirilmedi.")
print("✅ Test yalnızca final değerlendirmede kullanıldı.")


Creating file manifest:   0%|          | 0/35 [00:00<?, ?it/s]


FINAL EXPERIMENT AUDIT
✅ required_files_present: True
✅ data_accounting_valid: True
✅ source_video_isolation_valid: True
✅ sha256_split_isolation_valid: True
✅ test_protocol_valid: True
✅ output_separate_from_raw_data: True
✅ no_temporary_files_remaining: True
✅ frame_prediction_count_valid: True
✅ video_prediction_count_valid: True
✅ figure_quality_valid: True

DATA ACCOUNTING
----------------------------------------------------------------------
{'input_total': 3000, 'success': 2889, 'skipped': 111, 'error': 0}

SOURCE-VIDEO OVERLAPS
----------------------------------------------------------------------
Train-validation: 0
Train-test: 0
Validation-test: 0

HASH OVERLAPS
----------------------------------------------------------------------
Train-validation: 0
Train-test: 0
Validation-test: 0

FINAL RESULT
----------------------------------------------------------------------
Audit sonucu: ✅ PASS
Manifest dosya sayısı: 35
Nihai JSON: /content/drive/MyDrive/AISC DeepFake Çalışmaları